## **Installations, Imports, and Configurations**

In [2]:
# ============================================================
# Cell 0 — Verify/install inference-variant dependencies
# Run once at the beginning of the new notebook
# ============================================================

import sys
import subprocess
import importlib.util

REQUIRED_PACKAGES = {
    "torch": "torch",
    "transformers": "transformers<5",
    "peft": "peft",
    "datasets": "datasets",
    "pandas": "pandas",
    "numpy": "numpy",
    "sacrebleu": "sacrebleu[sentencepiece]",
    "sentencepiece": "sentencepiece",
    "tqdm": "tqdm",
    "sentence_transformers": "sentence-transformers",
}

missing_specs = []

for module_name, pip_spec in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(module_name) is None:
        missing_specs.append(pip_spec)

if missing_specs:
    print("Installing missing packages:")
    for spec in missing_specs:
        print(" -", spec)

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", *missing_specs]
    )
else:
    print("All required packages are already installed.")

print("\nPython:", sys.version)
print("Executable:", sys.executable)

All required packages are already installed.

Python: 3.11.15 (main, Jun 11 2026, 15:20:16) [GCC 14.3.0]
Executable: /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/bin/python


In [3]:
# ============================================================
# Cell 1 — Paths and common inference configuration
#
# All new experiments are stored under:
#   ~/alexandriax_mt_14d/inference_variants/
#
# No existing prediction or checkpoint folder is modified.
# ============================================================

import os
import gc
import re
import ast
import json
import time
import math
import random
import hashlib
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import sacrebleu

from tqdm.auto import tqdm
from IPython.display import display

from datasets import (
    load_dataset,
    get_dataset_config_names,
    get_dataset_split_names,
)

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 3407

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Project and model paths
# ------------------------------------------------------------

PROJECT_DIR = Path(
    os.environ.get(
        "AXMT_HOME",
        str(Path.home() / "alexandriax_mt_14d"),
    )
).expanduser()

BASE_MODEL_DIR = (
    PROJECT_DIR
    / "models"
    / "hf"
    / "NileChat-3B-Base"
)

EXPERIMENT_NAME = (
    "nilechat3b_alexandria_all14_"
    "context3_complete2shot_all_group_r16_alpha32_"
    "3epochs_beam4_nonquant_server5090_v1"
)

TRAINING_RUN_DIR = (
    PROJECT_DIR
    / "runs"
    / "nilechat3b_all14"
    / EXPERIMENT_NAME
)

CHECKPOINT_STEP = 16600
CHECKPOINT_PATH = TRAINING_RUN_DIR / f"checkpoint-{CHECKPOINT_STEP}"

# New independent experiment root
INFERENCE_VARIANTS_ROOT = PROJECT_DIR / "inference_variants"
INFERENCE_VARIANTS_ROOT.mkdir(parents=True, exist_ok=True)

# Shared caches used only by the new notebook
SHARED_CACHE_DIR = INFERENCE_VARIANTS_ROOT / "_shared_cache"
SHARED_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Dataset/evaluation settings
# ------------------------------------------------------------

DATASET_NAME = "UBC-NLP/alexandria"
OFFICIAL_SPLIT = "dev"

EXPECTED_DEV_TURNS = 12250
EXPECTED_DEV_COUNTRIES = 11

MAX_CONTEXT_TURNS = 3
MAX_SEQ_LENGTH = 2048
MAX_NEW_TOKENS = 120

N_FEW_SHOTS = 2
MAX_FEW_SHOT_EXAMPLE_CHARS = 450

# ------------------------------------------------------------
# Generation settings
# These reproduce the checkpoint sweep decoding.
# ------------------------------------------------------------

GEN_BATCH_SIZE = 2
SAVE_EVERY = 50

GENERATION_KWARGS = {
    "do_sample": False,
    "num_beams": 4,
    "num_return_sequences": 1,
    "length_penalty": 1.0,
    "early_stopping": True,
    "repetition_penalty": 1.05,
    "use_cache": True,
}

DECODE_TAG = "beam4"
PROMPT_VERSION = "inference_variants_training_style_marker_v1"

SYSTEM_MARKER = "### System:"
INSTRUCTION_MARKER = "### Instruction:"
RESPONSE_MARKER = "### Arabic translation:"

SYSTEM_PROMPT = (
    "You are a professional machine translation system. "
    "Translate the current English dialogue turn into natural dialectal Arabic. "
    "Use the provided training examples only as style and dialect guidance. "
    "Return only the translation, without explanation."
)

# English retrieval model for the retrieved-shot experiment
RETRIEVER_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
RETRIEVER_BATCH_SIZE = 256

# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

if not PROJECT_DIR.exists():
    raise FileNotFoundError(f"Project directory not found: {PROJECT_DIR}")

if not BASE_MODEL_DIR.exists():
    raise FileNotFoundError(f"Base model not found: {BASE_MODEL_DIR}")

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT_PATH}")

if not (CHECKPOINT_PATH / "adapter_config.json").exists():
    raise FileNotFoundError(
        f"adapter_config.json missing from checkpoint: {CHECKPOINT_PATH}"
    )

adapter_weight_candidates = [
    CHECKPOINT_PATH / "adapter_model.safetensors",
    CHECKPOINT_PATH / "adapter_model.bin",
]

if not any(p.exists() for p in adapter_weight_candidates):
    raise FileNotFoundError(
        f"No adapter weights found inside: {CHECKPOINT_PATH}"
    )

print("==============================")
print("Inference variants configuration")
print("==============================")
print("Project:", PROJECT_DIR)
print("Base model:", BASE_MODEL_DIR)
print("Checkpoint:", CHECKPOINT_PATH)
print("Checkpoint step:", CHECKPOINT_STEP)
print("Variants root:", INFERENCE_VARIANTS_ROOT)
print("Shared cache:", SHARED_CACHE_DIR)
print("Expected dev turns:", EXPECTED_DEV_TURNS)
print("Decode:", DECODE_TAG)
print("Generation kwargs:", GENERATION_KWARGS)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())

Failed to load /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so


Inference variants configuration
Project: /home/mabdallah/alexandriax_mt_14d
Base model: /home/mabdallah/alexandriax_mt_14d/models/hf/NileChat-3B-Base
Checkpoint: /home/mabdallah/alexandriax_mt_14d/runs/nilechat3b_all14/nilechat3b_alexandria_all14_context3_complete2shot_all_group_r16_alpha32_3epochs_beam4_nonquant_server5090_v1/checkpoint-16600
Checkpoint step: 16600
Variants root: /home/mabdallah/alexandriax_mt_14d/inference_variants
Shared cache: /home/mabdallah/alexandriax_mt_14d/inference_variants/_shared_cache
Expected dev turns: 12250
Decode: beam4
Generation kwargs: {'do_sample': False, 'num_beams': 4, 'num_return_sequences': 1, 'length_penalty': 1.0, 'early_stopping': True, 'repetition_penalty': 1.05, 'use_cache': True}
CUDA available: True
GPU: NVIDIA GeForce RTX 5090
BF16 supported: True


## **Build the official development dataframe and training retrieval pool**

In [4]:
# ============================================================
# Cell 2 — Prepare official DEV and training few-shot pool
#
# This cell:
#   1. Loads the actual public DEV split.
#   2. Pairs English and dialectal turns correctly.
#   3. Preserves turn_order, speaker, and direction.
#   4. Preserves participants and translator/reviewer IDs.
#   5. Keeps target references for exact local scoring.
#   6. Builds the training pool used by random/retrieved shots.
#   7. Caches the prepared data resumably.
# ============================================================

PREPARED_CACHE_VERSION = "paired_dev_train_v3"

PREPARED_CACHE_DIR = (
    SHARED_CACHE_DIR / PREPARED_CACHE_VERSION
)
PREPARED_CACHE_DIR.mkdir(parents=True, exist_ok=True)

DEV_CACHE_PATH = PREPARED_CACHE_DIR / "official_dev_df.pkl"
TRAIN_CACHE_PATH = PREPARED_CACHE_DIR / "train_fewshot_pool_df.pkl"
CACHE_REPORT_PATH = PREPARED_CACHE_DIR / "prepared_data_report.json"

# ------------------------------------------------------------
# Normalization helpers
# ------------------------------------------------------------

def to_plain(x):
    if isinstance(x, np.ndarray):
        return [to_plain(v) for v in x.tolist()]

    if isinstance(x, np.generic):
        return x.item()

    if isinstance(x, tuple):
        return [to_plain(v) for v in x]

    if isinstance(x, list):
        return [to_plain(v) for v in x]

    if isinstance(x, dict):
        return {str(k): to_plain(v) for k, v in x.items()}

    return x


def scalar_text(x):
    x = to_plain(x)

    if x is None:
        return ""

    if isinstance(x, (list, dict)):
        return str(x).strip()

    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass

    return str(x).strip()


def normalize_turn_list(value):
    """
    Convert:
      - list[dict]
      - numpy array of dicts
      - dict of parallel lists
    into list[dict].
    """

    value = to_plain(value)

    if value is None:
        return []

    if isinstance(value, list):
        normalized = []

        for item in value:
            item = to_plain(item)

            if isinstance(item, dict):
                normalized.append(item)
            elif item is not None:
                normalized.append({"text": scalar_text(item)})

        return normalized

    if isinstance(value, dict):
        list_lengths = [
            len(v)
            for v in value.values()
            if isinstance(to_plain(v), list)
        ]

        if not list_lengths:
            return []

        n = max(list_lengths)
        rows = []

        for i in range(n):
            item = {}

            for key, values in value.items():
                values = to_plain(values)

                if isinstance(values, list):
                    item[key] = values[i] if i < len(values) else None
                else:
                    item[key] = values

            rows.append(item)

        return rows

    return []


def turn_field(turn, keys, default=""):
    turn = to_plain(turn)

    if not isinstance(turn, dict):
        return default

    for key in keys:
        if key in turn and turn[key] is not None:
            value = scalar_text(turn[key])

            if value:
                return value

    return default


def turn_text(turn):
    return turn_field(
        turn,
        [
            "text",
            "sentence",
            "utterance",
            "content",
            "value",
            "english",
            "source_text",
            "target_arabic",
            "translation",
        ],
        default="",
    )


def extract_turn_order(turn, fallback_index):
    raw = turn_field(
        turn,
        ["turn_order", "turn_id", "order", "idx", "index"],
        default="",
    )

    if raw:
        try:
            return int(raw)
        except Exception:
            pass

    return int(fallback_index + 1)


def dataframe_id_hash(df):
    h = hashlib.sha256()

    for value in df["source_id"].astype(str).tolist():
        h.update(value.encode("utf-8"))
        h.update(b"\n")

    return h.hexdigest()


# ------------------------------------------------------------
# Flatten one Alexandria split
# ------------------------------------------------------------

def flatten_alexandria_split(
    dataset,
    config_name,
    split_name,
    include_context,
):
    records = []

    for conv_index, raw_row in enumerate(dataset):
        row = to_plain(dict(raw_row))

        conv_id = scalar_text(
            row.get(
                "conv_id",
                row.get(
                    "conversation_id",
                    f"{config_name}_{split_name}_{conv_index}",
                ),
            )
        )

        country = scalar_text(row.get("country", config_name)) or config_name
        dialect = scalar_text(row.get("dialect", ""))
        domain = scalar_text(row.get("domain", ""))
        participants = scalar_text(row.get("participants", ""))

        translator_id = scalar_text(row.get("translator_id", ""))
        reviewer_id = scalar_text(row.get("reviewer_id", ""))

        english_turns = normalize_turn_list(
            row.get("english_conversation", [])
        )

        arabic_turns = normalize_turn_list(
            row.get("dialectal_conversation", [])
        )

        n_turns = min(len(english_turns), len(arabic_turns))

        if n_turns == 0:
            continue

        for turn_index in range(n_turns):
            english_turn = english_turns[turn_index]
            arabic_turn = arabic_turns[turn_index]

            source_text = turn_text(english_turn)
            target_arabic = turn_text(arabic_turn)

            if not source_text or not target_arabic:
                continue

            turn_order = extract_turn_order(
                english_turn,
                fallback_index=turn_index,
            )

            arabic_turn_order = extract_turn_order(
                arabic_turn,
                fallback_index=turn_index,
            )

            if turn_order != arabic_turn_order:
                raise RuntimeError(
                    "English/Arabic turn-order mismatch: "
                    f"{config_name}/{split_name}/{conv_id} "
                    f"English={turn_order}, Arabic={arabic_turn_order}"
                )

            speaker = turn_field(
                english_turn,
                ["speaker", "role", "speaker_role", "participant"],
                default="",
            )

            gender_direction = turn_field(
                english_turn,
                [
                    "direction",
                    "gender_direction",
                    "speaker_addressee_gender",
                ],
                default="",
            )

            previous_english_turns = []

            if include_context:
                previous_start = max(
                    0,
                    turn_index - MAX_CONTEXT_TURNS,
                )

                for previous_index in range(
                    previous_start,
                    turn_index,
                ):
                    previous_turn = english_turns[previous_index]

                    previous_english_turns.append({
                        "turn_order": extract_turn_order(
                            previous_turn,
                            fallback_index=previous_index,
                        ),
                        "speaker": turn_field(
                            previous_turn,
                            [
                                "speaker",
                                "role",
                                "speaker_role",
                                "participant",
                            ],
                            default="",
                        ),
                        "direction": turn_field(
                            previous_turn,
                            [
                                "direction",
                                "gender_direction",
                                "speaker_addressee_gender",
                            ],
                            default="",
                        ),
                        "text": turn_text(previous_turn),
                    })

            source_id = (
                f"{config_name}_{split_name}_"
                f"{conv_id}_{turn_order}"
            )

            records.append({
                "source_id": source_id,
                "config": config_name,
                "country": country,
                "split": split_name,
                "conversation_id": conv_id,
                "turn_order": int(turn_order),
                "turn_id": int(turn_order),

                "dialect": dialect,
                "domain": domain,
                "participants": participants,

                "speaker": speaker,
                "gender_direction": gender_direction,
                "previous_english_turns": previous_english_turns,

                "source_text": source_text,
                "target_arabic": target_arabic,
                "reference_arabic": target_arabic,

                "translator_id": translator_id,
                "reviewer_id": reviewer_id,
            })

    return records


# ------------------------------------------------------------
# Load from cache or rebuild
# ------------------------------------------------------------

if DEV_CACHE_PATH.exists() and TRAIN_CACHE_PATH.exists():
    print("Loading prepared data from cache...")

    official_dev_df = pd.read_pickle(DEV_CACHE_PATH)
    train_fewshot_df = pd.read_pickle(TRAIN_CACHE_PATH)

else:
    print("Prepared cache not found. Loading Alexandria dataset...")

    available_configs = get_dataset_config_names(DATASET_NAME)

    split_map = {
        config_name: get_dataset_split_names(
            DATASET_NAME,
            config_name,
        )
        for config_name in available_configs
    }

    OFFICIAL_DEV_CONFIGS = sorted([
        config_name
        for config_name in available_configs
        if OFFICIAL_SPLIT in split_map[config_name]
    ])

    TRAIN_CONFIGS = sorted([
        config_name
        for config_name in available_configs
        if "train" in split_map[config_name]
    ])

    print("Official DEV configs:", OFFICIAL_DEV_CONFIGS)
    print("Train configs:", TRAIN_CONFIGS)

    dev_records = []
    train_records = []

    for config_name in OFFICIAL_DEV_CONFIGS:
        print(f"\nLoading {config_name}/dev ...")

        dev_dataset = load_dataset(
            DATASET_NAME,
            config_name,
            split="dev",
        )

        config_records = flatten_alexandria_split(
            dev_dataset,
            config_name=config_name,
            split_name="dev",
            include_context=True,
        )

        print(
            config_name,
            "dev turns:",
            len(config_records),
        )

        dev_records.extend(config_records)

    for config_name in TRAIN_CONFIGS:
        print(f"\nLoading {config_name}/train ...")

        train_dataset = load_dataset(
            DATASET_NAME,
            config_name,
            split="train",
        )

        config_records = flatten_alexandria_split(
            train_dataset,
            config_name=config_name,
            split_name="train",
            include_context=False,
        )

        print(
            config_name,
            "train turns:",
            len(config_records),
        )

        train_records.extend(config_records)

    official_dev_df = pd.DataFrame(dev_records)
    train_fewshot_df = pd.DataFrame(train_records)

    official_dev_df = official_dev_df.sort_values(
        ["config", "conversation_id", "turn_order"]
    ).reset_index(drop=True)

    train_fewshot_df = train_fewshot_df.reset_index(drop=True)

    official_dev_df.to_pickle(DEV_CACHE_PATH)
    train_fewshot_df.to_pickle(TRAIN_CACHE_PATH)

    print("\nSaved prepared data cache:")
    print("DEV:", DEV_CACHE_PATH)
    print("TRAIN:", TRAIN_CACHE_PATH)

# ------------------------------------------------------------
# Final normalization
# ------------------------------------------------------------

official_dev_df["source_id"] = (
    official_dev_df["source_id"].astype(str)
)

official_dev_df["conversation_id"] = (
    official_dev_df["conversation_id"].astype(str)
)

official_dev_df["config"] = (
    official_dev_df["config"].astype(str)
)

official_dev_df["turn_order"] = pd.to_numeric(
    official_dev_df["turn_order"],
    errors="raise",
).astype(int)

official_dev_df["source_text"] = (
    official_dev_df["source_text"]
    .fillna("")
    .astype(str)
    .str.strip()
)

official_dev_df["reference_arabic"] = (
    official_dev_df["reference_arabic"]
    .fillna("")
    .astype(str)
    .str.strip()
)

official_dev_df["_dev_row_idx"] = np.arange(
    len(official_dev_df),
    dtype=np.int64,
)

for column in [
    "source_id",
    "config",
    "conversation_id",
    "dialect",
    "domain",
    "participants",
    "speaker",
    "gender_direction",
    "source_text",
    "target_arabic",
    "reference_arabic",
    "translator_id",
    "reviewer_id",
]:
    if column in train_fewshot_df.columns:
        train_fewshot_df[column] = (
            train_fewshot_df[column]
            .fillna("")
            .astype(str)
        )

OFFICIAL_DEV_CONFIGS = sorted(
    official_dev_df["config"].unique().tolist()
)

TRAIN_CONFIGS = sorted(
    train_fewshot_df["config"].unique().tolist()
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("\n==============================")
print("Prepared-data validation")
print("==============================")
print("Official DEV countries:", OFFICIAL_DEV_CONFIGS)
print("Official DEV turns:", len(official_dev_df))
print("Train turns:", len(train_fewshot_df))
print(
    "DEV conversations:",
    official_dev_df[
        ["config", "conversation_id"]
    ].drop_duplicates().shape[0],
)
print(
    "Duplicate DEV source_id:",
    int(official_dev_df["source_id"].duplicated().sum()),
)
print(
    "Empty DEV sources:",
    int((official_dev_df["source_text"] == "").sum()),
)
print(
    "Empty DEV references:",
    int((official_dev_df["reference_arabic"] == "").sum()),
)

if len(official_dev_df) != EXPECTED_DEV_TURNS:
    raise RuntimeError(
        f"Expected {EXPECTED_DEV_TURNS} DEV turns, "
        f"found {len(official_dev_df)}"
    )

if len(OFFICIAL_DEV_CONFIGS) != EXPECTED_DEV_COUNTRIES:
    raise RuntimeError(
        f"Expected {EXPECTED_DEV_COUNTRIES} DEV countries, "
        f"found {len(OFFICIAL_DEV_CONFIGS)}: "
        f"{OFFICIAL_DEV_CONFIGS}"
    )

if official_dev_df["source_id"].duplicated().any():
    raise RuntimeError("Duplicate DEV source_id found.")

if int((official_dev_df["source_text"] == "").sum()):
    raise RuntimeError("Empty DEV source text found.")

if int((official_dev_df["reference_arabic"] == "").sum()):
    raise RuntimeError("Empty DEV reference found.")

expected_debug_keys = {
    ("EG", "B0-1-0-183", 1),
    ("EG", "B0-1-0-183", 2),
}

actual_keys = set(zip(
    official_dev_df["config"].astype(str),
    official_dev_df["conversation_id"].astype(str),
    official_dev_df["turn_order"].astype(int),
))

missing_debug_keys = expected_debug_keys - actual_keys

if missing_debug_keys:
    raise RuntimeError(
        f"Missing scorer debug keys: {missing_debug_keys}"
    )

country_counts = (
    official_dev_df
    .groupby("config")
    .agg(
        turns=("source_id", "count"),
        conversations=("conversation_id", "nunique"),
        min_turn=("turn_order", "min"),
        max_turn=("turn_order", "max"),
    )
    .reset_index()
)

print("\nOfficial DEV country counts:")
display(country_counts)

print("\nTraining pool counts:")
display(
    train_fewshot_df["config"]
    .value_counts()
    .sort_index()
    .rename_axis("config")
    .reset_index(name="turns")
)

debug_df = official_dev_df[
    (official_dev_df["config"] == "EG")
    & (
        official_dev_df["conversation_id"]
        == "B0-1-0-183"
    )
].copy()

print("\nDebug conversation with restored metadata:")
display(
    debug_df[
        [
            "source_id",
            "config",
            "conversation_id",
            "turn_order",
            "speaker",
            "gender_direction",
            "previous_english_turns",
            "source_text",
            "reference_arabic",
        ]
    ]
)

prepared_report = {
    "cache_version": PREPARED_CACHE_VERSION,
    "dataset_name": DATASET_NAME,
    "official_split": OFFICIAL_SPLIT,
    "official_dev_configs": OFFICIAL_DEV_CONFIGS,
    "train_configs": TRAIN_CONFIGS,
    "dev_turns": int(len(official_dev_df)),
    "train_turns": int(len(train_fewshot_df)),
    "dev_source_id_hash": dataframe_id_hash(
        official_dev_df
    ),
    "train_source_id_hash": dataframe_id_hash(
        train_fewshot_df
    ),
}

with open(
    CACHE_REPORT_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        prepared_report,
        f,
        ensure_ascii=False,
        indent=2,
    )

print("\nPrepared-data report:", CACHE_REPORT_PATH)
print("✅ Official DEV and training pool are ready.")

Loading prepared data from cache...

Prepared-data validation
Official DEV countries: ['EG', 'JO', 'LB', 'MA', 'MR', 'OM', 'PS', 'SA', 'SY', 'TN', 'YE']
Official DEV turns: 12250
Train turns: 66480
DEV conversations: 3963
Duplicate DEV source_id: 0
Empty DEV sources: 0
Empty DEV references: 0

Official DEV country counts:


,config,turns,conversations,min_turn,max_turn
0,EG,1113,352,1,4
1,JO,1113,352,1,5
2,LB,1118,371,1,4
3,MA,1110,354,1,5
4,MR,1114,352,1,4
5,OM,1109,381,1,4
6,PS,1110,352,1,4
7,SA,1110,363,1,4
8,SY,1119,347,1,4
9,TN,1116,378,1,4



Training pool counts:


,config,turns
0,EG,3108
1,JO,5501
2,LB,8906
3,MA,2573
4,MR,5515
5,OM,6280
6,PS,14933
7,SA,8470
8,SY,6071
9,TN,2034



Debug conversation with restored metadata:


,source_id,config,conversation_id,turn_order,speaker,gender_direction,previous_english_turns,source_text,reference_arabic
0,EG_dev_B0-1-0-183_1,EG,B0-1-0-183,1,Waterresourcemanager,female -> female,[],"Good news, the ministry has finally approved t...",أخبار حلوة، الوزارة أخيرا وافقت على الميزانية ...
1,EG_dev_B0-1-0-183_2,EG,B0-1-0-183,2,Seniorengineer,female -> female,"[{'turn_order': 1, 'speaker': 'Waterresourcema...",Thank God! That is a huge relief. When can we ...,الحمد لله! ديه حاجة مريحة قوي. إمتى ممكن نبدأ ...
2,EG_dev_B0-1-0-183_3,EG,B0-1-0-183,3,Waterresourcemanager,female -> female,"[{'turn_order': 1, 'speaker': 'Waterresourcema...",I'm pushing to get the paperwork started next ...,أنا بضغط علشان نبدأ الورق الأسبوع الجاي. عندنا...



Prepared-data report: /home/mabdallah/alexandriax_mt_14d/inference_variants/_shared_cache/paired_dev_train_v3/prepared_data_report.json
✅ Official DEV and training pool are ready.


## **Shared resumable inference, scoring, and submission functions**

In [5]:
# ============================================================
# Cell 3 — Shared inference-variant engine
#
# Every experiment cell below calls:
#     run_inference_variant(VARIANT_SPEC)
#
# Each call:
#   - creates an independent variant folder
#   - selects its own few-shot policy
#   - generates resumably
#   - saves every SAVE_EVERY rows
#   - scores with official spBLEU and chrF++
#   - displays overall and per-country metrics
#   - writes predictions.jsonl
#   - writes submission_predictions.zip
# ============================================================

# ------------------------------------------------------------
# Few-shot pools
# ------------------------------------------------------------

FEW_SHOT_COLUMNS = [
    "source_id",
    "config",
    "conversation_id",
    "dialect",
    "domain",
    "participants",
    "speaker",
    "gender_direction",
    "source_text",
    "target_arabic",
    "translator_id",
    "reviewer_id",
]

missing_train_columns = [
    column
    for column in FEW_SHOT_COLUMNS
    if column not in train_fewshot_df.columns
]

if missing_train_columns:
    raise ValueError(
        f"Training pool missing columns: "
        f"{missing_train_columns}"
    )

train_shot_pool = train_fewshot_df[
    FEW_SHOT_COLUMNS
].copy().reset_index(drop=True)

train_shot_pool["fewshot_total_chars"] = (
    train_shot_pool["source_text"].astype(str).str.len()
    + train_shot_pool["target_arabic"].astype(str).str.len()
)

short_train_shot_pool = train_shot_pool[
    train_shot_pool["fewshot_total_chars"]
    <= MAX_FEW_SHOT_EXAMPLE_CHARS
].copy().reset_index(drop=True)

random_pool_by_config = {
    key: group
    for key, group in train_shot_pool.groupby(
        "config",
        sort=False,
    )
}

random_short_by_config = {
    key: group
    for key, group in short_train_shot_pool.groupby(
        "config",
        sort=False,
    )
}

random_pool_by_config_domain = {
    key: group
    for key, group in train_shot_pool.groupby(
        ["config", "domain"],
        sort=False,
    )
}

random_short_by_config_domain = {
    key: group
    for key, group in short_train_shot_pool.groupby(
        ["config", "domain"],
        sort=False,
    )
}

# Retrieval always starts from short examples.
retrieval_train_pool = (
    short_train_shot_pool
    .copy()
    .reset_index(drop=True)
)

retrieval_indices_by_config = {
    key: group.index.to_numpy(dtype=np.int64)
    for key, group in retrieval_train_pool.groupby(
        "config",
        sort=False,
    )
}

retrieval_indices_by_config_domain = {
    key: group.index.to_numpy(dtype=np.int64)
    for key, group in retrieval_train_pool.groupby(
        ["config", "domain"],
        sort=False,
    )
}

# ------------------------------------------------------------
# General helpers
# ------------------------------------------------------------

def stable_seed_from_id(source_id, base_seed=SEED):
    raw = f"{source_id}_{base_seed}".encode("utf-8")
    return int(hashlib.md5(raw).hexdigest()[:8], 16)


def safe_string(value):
    if value is None:
        return ""

    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass

    return str(value).strip()


def variant_fingerprint(spec):
    payload = {
        "spec": spec,
        "prompt_version": PROMPT_VERSION,
        "checkpoint_path": str(CHECKPOINT_PATH),
        "checkpoint_step": CHECKPOINT_STEP,
        "base_model_dir": str(BASE_MODEL_DIR),
        "max_context_turns": MAX_CONTEXT_TURNS,
        "max_seq_length": MAX_SEQ_LENGTH,
        "max_new_tokens": MAX_NEW_TOKENS,
        "generation_kwargs": GENERATION_KWARGS,
        "system_prompt": SYSTEM_PROMPT,
        "system_marker": SYSTEM_MARKER,
        "instruction_marker": INSTRUCTION_MARKER,
        "response_marker": RESPONSE_MARKER,
    }

    serialized = json.dumps(
        payload,
        ensure_ascii=False,
        sort_keys=True,
    )

    return hashlib.sha256(
        serialized.encode("utf-8")
    ).hexdigest()


# ------------------------------------------------------------
# Random-shot selection reproducing the training policy
# ------------------------------------------------------------

def select_random_two_shots(row, n=N_FEW_SHOTS):
    config_name = safe_string(row.get("config", ""))
    domain = safe_string(row.get("domain", ""))
    source_id = safe_string(row.get("source_id", ""))

    candidate_pools = [
        random_short_by_config_domain.get(
            (config_name, domain)
        ),
        random_short_by_config.get(config_name),
        short_train_shot_pool,
        random_pool_by_config_domain.get(
            (config_name, domain)
        ),
        random_pool_by_config.get(config_name),
        train_shot_pool,
    ]

    random_state = stable_seed_from_id(source_id)

    for candidate_pool in candidate_pools:
        if candidate_pool is None:
            continue

        if len(candidate_pool) == 0:
            continue

        sample_n = min(n, len(candidate_pool))

        sampled = candidate_pool.sample(
            n=sample_n,
            random_state=random_state,
        )

        return sampled[
            FEW_SHOT_COLUMNS
        ].to_dict("records")

    return []


# ------------------------------------------------------------
# Semantic retrieval cache
# ------------------------------------------------------------

RETRIEVAL_CACHE_DIR = (
    SHARED_CACHE_DIR
    / "semantic_retrieval_all_minilm_l6_v2"
)
RETRIEVAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

RETRIEVAL_TRAIN_EMBEDDINGS = (
    RETRIEVAL_CACHE_DIR
    / "train_embeddings.npy"
)

RETRIEVAL_DEV_EMBEDDINGS = (
    RETRIEVAL_CACHE_DIR
    / "dev_embeddings.npy"
)

RETRIEVAL_MANIFEST_PATH = (
    RETRIEVAL_CACHE_DIR
    / "retrieval_manifest.json"
)


def get_semantic_retrieval_embeddings():
    train_hash = dataframe_id_hash(
        retrieval_train_pool
    )

    dev_hash = dataframe_id_hash(
        official_dev_df
    )

    expected_manifest = {
        "retriever_model": RETRIEVER_MODEL_NAME,
        "train_rows": int(len(retrieval_train_pool)),
        "dev_rows": int(len(official_dev_df)),
        "train_hash": train_hash,
        "dev_hash": dev_hash,
    }

    cache_valid = False

    if (
        RETRIEVAL_MANIFEST_PATH.exists()
        and RETRIEVAL_TRAIN_EMBEDDINGS.exists()
        and RETRIEVAL_DEV_EMBEDDINGS.exists()
    ):
        with open(
            RETRIEVAL_MANIFEST_PATH,
            "r",
            encoding="utf-8",
        ) as f:
            existing_manifest = json.load(f)

        cache_valid = (
            existing_manifest == expected_manifest
        )

    if cache_valid:
        print("Loading cached semantic embeddings...")

        train_embeddings = np.load(
            RETRIEVAL_TRAIN_EMBEDDINGS,
            mmap_mode="r",
        )

        dev_embeddings = np.load(
            RETRIEVAL_DEV_EMBEDDINGS,
            mmap_mode="r",
        )

        return train_embeddings, dev_embeddings

    print("Building semantic retrieval embeddings...")

    from sentence_transformers import SentenceTransformer

    retrieval_device = (
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    retrieval_model = SentenceTransformer(
        RETRIEVER_MODEL_NAME,
        device=retrieval_device,
    )

    train_texts = (
        retrieval_train_pool["source_text"]
        .fillna("")
        .astype(str)
        .tolist()
    )

    dev_texts = (
        official_dev_df["source_text"]
        .fillna("")
        .astype(str)
        .tolist()
    )

    train_embeddings = retrieval_model.encode(
        train_texts,
        batch_size=RETRIEVER_BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)

    dev_embeddings = retrieval_model.encode(
        dev_texts,
        batch_size=RETRIEVER_BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)

    np.save(
        RETRIEVAL_TRAIN_EMBEDDINGS,
        train_embeddings,
    )

    np.save(
        RETRIEVAL_DEV_EMBEDDINGS,
        dev_embeddings,
    )

    with open(
        RETRIEVAL_MANIFEST_PATH,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            expected_manifest,
            f,
            ensure_ascii=False,
            indent=2,
        )

    del retrieval_model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    train_embeddings = np.load(
        RETRIEVAL_TRAIN_EMBEDDINGS,
        mmap_mode="r",
    )

    dev_embeddings = np.load(
        RETRIEVAL_DEV_EMBEDDINGS,
        mmap_mode="r",
    )

    return train_embeddings, dev_embeddings


def select_retrieved_two_shots(
    row,
    train_embeddings,
    dev_embeddings,
    n=N_FEW_SHOTS,
):
    config_name = safe_string(row.get("config", ""))
    domain = safe_string(row.get("domain", ""))
    dialect = safe_string(row.get("dialect", ""))
    speaker = safe_string(row.get("speaker", ""))
    direction = safe_string(
        row.get("gender_direction", "")
    )

    dev_row_index = int(row["_dev_row_idx"])

    candidate_indices = (
        retrieval_indices_by_config_domain.get(
            (config_name, domain)
        )
    )

    if (
        candidate_indices is None
        or len(candidate_indices) < n
    ):
        candidate_indices = (
            retrieval_indices_by_config.get(
                config_name
            )
        )

    if (
        candidate_indices is None
        or len(candidate_indices) == 0
    ):
        return select_random_two_shots(row, n=n)

    query_embedding = np.asarray(
        dev_embeddings[dev_row_index],
        dtype=np.float32,
    )

    candidate_embeddings = np.asarray(
        train_embeddings[candidate_indices],
        dtype=np.float32,
    )

    scores = candidate_embeddings @ query_embedding

    candidate_meta = retrieval_train_pool.iloc[
        candidate_indices
    ]

    if dialect:
        scores = scores + (
            candidate_meta["dialect"]
            .astype(str)
            .to_numpy()
            == dialect
        ).astype(np.float32) * 0.05

    if direction:
        scores = scores + (
            candidate_meta["gender_direction"]
            .astype(str)
            .to_numpy()
            == direction
        ).astype(np.float32) * 0.03

    if speaker:
        scores = scores + (
            candidate_meta["speaker"]
            .astype(str)
            .to_numpy()
            == speaker
        ).astype(np.float32) * 0.02

    ranked_positions = np.argsort(-scores)

    selected_records = []
    used_conversations = set()

    # First pass: prefer different training conversations.
    for position in ranked_positions:
        pool_index = int(candidate_indices[position])

        candidate = retrieval_train_pool.iloc[
            pool_index
        ]

        conversation_key = (
            safe_string(candidate["config"]),
            safe_string(candidate["conversation_id"]),
        )

        if conversation_key in used_conversations:
            continue

        selected_records.append(
            candidate[
                FEW_SHOT_COLUMNS
            ].to_dict()
        )

        used_conversations.add(conversation_key)

        if len(selected_records) >= n:
            break

    # Fallback if there were not enough unique conversations.
    if len(selected_records) < n:
        selected_ids = {
            safe_string(x["source_id"])
            for x in selected_records
        }

        for position in ranked_positions:
            pool_index = int(candidate_indices[position])

            candidate = retrieval_train_pool.iloc[
                pool_index
            ]

            candidate_source_id = safe_string(
                candidate["source_id"]
            )

            if candidate_source_id in selected_ids:
                continue

            selected_records.append(
                candidate[
                    FEW_SHOT_COLUMNS
                ].to_dict()
            )

            selected_ids.add(candidate_source_id)

            if len(selected_records) >= n:
                break

    return selected_records[:n]


# ------------------------------------------------------------
# Variant dataframe preparation
# ------------------------------------------------------------

def prepare_variant_dataframe(
    spec,
    variant_dir,
    fingerprint,
):
    variant_df = official_dev_df.copy()

    shots_cache_path = (
        variant_dir / "selected_few_shots.pkl"
    )

    shot_mode = spec["shot_mode"]

    if shot_mode == "none":
        variant_df["few_shot_examples"] = [
            []
            for _ in range(len(variant_df))
        ]

        return variant_df

    if shots_cache_path.exists():
        cached_shots = pd.read_pickle(
            shots_cache_path
        )

        if (
            "variant_fingerprint"
            not in cached_shots.columns
        ):
            raise RuntimeError(
                f"Old/incompatible shot cache: "
                f"{shots_cache_path}"
            )

        cached_fingerprints = set(
            cached_shots[
                "variant_fingerprint"
            ].astype(str)
        )

        if cached_fingerprints != {fingerprint}:
            raise RuntimeError(
                "Few-shot cache fingerprint mismatch. "
                "Use a new variant_name."
            )

        shot_map = dict(zip(
            cached_shots["source_id"].astype(str),
            cached_shots["few_shot_examples"],
        ))

        missing_shot_ids = (
            set(variant_df["source_id"].astype(str))
            - set(shot_map)
        )

        if missing_shot_ids:
            raise RuntimeError(
                "Few-shot cache is incomplete. "
                f"Missing={len(missing_shot_ids)}"
            )

        variant_df["few_shot_examples"] = (
            variant_df["source_id"]
            .astype(str)
            .map(shot_map)
        )

        print(
            "Loaded cached shot selections:",
            shots_cache_path,
        )

        return variant_df

    selected_shots = []

    if shot_mode == "random":
        print("Selecting deterministic random shots...")

        for _, row in tqdm(
            variant_df.iterrows(),
            total=len(variant_df),
            desc="random two-shot selection",
        ):
            selected_shots.append(
                select_random_two_shots(
                    row.to_dict(),
                    n=N_FEW_SHOTS,
                )
            )

    elif shot_mode == "retrieved":
        train_embeddings, dev_embeddings = (
            get_semantic_retrieval_embeddings()
        )

        print("Selecting semantically retrieved shots...")

        for _, row in tqdm(
            variant_df.iterrows(),
            total=len(variant_df),
            desc="retrieved two-shot selection",
        ):
            selected_shots.append(
                select_retrieved_two_shots(
                    row.to_dict(),
                    train_embeddings=train_embeddings,
                    dev_embeddings=dev_embeddings,
                    n=N_FEW_SHOTS,
                )
            )

        del train_embeddings
        del dev_embeddings
        gc.collect()

    else:
        raise ValueError(
            f"Unknown shot_mode: {shot_mode}"
        )

    variant_df["few_shot_examples"] = (
        selected_shots
    )

    shot_cache_df = pd.DataFrame({
        "source_id": variant_df["source_id"].astype(str),
        "few_shot_examples": selected_shots,
        "variant_fingerprint": fingerprint,
    })

    shot_cache_df.to_pickle(shots_cache_path)

    print("Saved shot selections:", shots_cache_path)

    return variant_df


# ------------------------------------------------------------
# Prompt building
# ------------------------------------------------------------

def build_context_for_variant(row, spec):
    previous_turns = to_plain(
        row.get("previous_english_turns", [])
    )

    if not previous_turns:
        return "No previous context."

    lines = []

    for index, previous_turn in enumerate(
        previous_turns,
        start=1,
    ):
        if isinstance(previous_turn, dict):
            previous_text = safe_string(
                previous_turn.get("text", "")
            )

            previous_speaker = safe_string(
                previous_turn.get("speaker", "")
            )
        else:
            previous_text = safe_string(
                previous_turn
            )

            previous_speaker = ""

        if not previous_text:
            continue

        if (
            spec["use_previous_speakers"]
            and previous_speaker
        ):
            lines.append(
                f"{index}. "
                f"{previous_speaker}: "
                f"{previous_text}"
            )
        else:
            lines.append(
                f"{index}. {previous_text}"
            )

    return (
        "\n".join(lines)
        if lines
        else "No previous context."
    )


def build_metadata_for_variant(row, spec):
    fields = [
        ("Country/config", row.get("config", "")),
        ("Target dialect", row.get("dialect", "")),
        ("Domain", row.get("domain", "")),
    ]

    if spec["use_participants"]:
        fields.append(
            (
                "Persona/Roles",
                row.get("participants", ""),
            )
        )

    fields.append(
        (
            "Current speaker",
            row.get("speaker", ""),
        )
    )

    if spec["use_direction"]:
        fields.append(
            (
                "Speaker-to-addressee gender direction",
                row.get("gender_direction", ""),
            )
        )

    lines = []

    for label, value in fields:
        value = safe_string(value)

        if value:
            lines.append(f"{label}: {value}")

    return (
        "\n".join(lines)
        if lines
        else "No metadata."
    )


def build_few_shot_block_for_variant(row):
    examples = to_plain(
        row.get("few_shot_examples", [])
    )

    if not examples:
        return "No examples available."

    blocks = []

    for example_index, example in enumerate(
        examples,
        start=1,
    ):
        config_name = safe_string(
            example.get("config", "")
        )

        dialect = safe_string(
            example.get("dialect", "")
        )

        domain = safe_string(
            example.get("domain", "")
        )

        metadata_parts = []

        if config_name:
            metadata_parts.append(
                f"config={config_name}"
            )

        if dialect:
            metadata_parts.append(
                f"dialect={dialect}"
            )

        if domain:
            metadata_parts.append(
                f"domain={domain}"
            )

        metadata_line = (
            ", ".join(metadata_parts)
            if metadata_parts
            else "no metadata"
        )

        example_source = safe_string(
            example.get("source_text", "")
        )

        example_target = safe_string(
            example.get("target_arabic", "")
        )

        blocks.append(
            f"""Example {example_index} ({metadata_line})
English:
{example_source}

Arabic:
{example_target}"""
        )

    return "\n\n".join(blocks)


def make_user_prompt_for_variant(row, spec):
    context = build_context_for_variant(
        row,
        spec,
    )

    metadata = build_metadata_for_variant(
        row,
        spec,
    )

    few_shots = build_few_shot_block_for_variant(
        row
    )

    return f"""Task:
Translate the current English dialogue turn into the target dialectal Arabic variety.

Few-shot training examples:
{few_shots}

Metadata:
{metadata}

Previous English dialogue context:
{context}

Current English turn:
{row["source_text"]}

Rules:
- Preserve the meaning exactly.
- Use the target local dialect, not Modern Standard Arabic unless it is natural in context.
- Follow the dialect/style pattern shown in the few-shot examples when relevant.
- Do not copy the few-shot examples.
- Preserve names, numbers, named entities, and technical terms when appropriate.
- Keep the tone appropriate for the speaker and domain.
- Return only the Arabic translation."""


def format_generation_prompt(row, spec):
    user_text = make_user_prompt_for_variant(
        row,
        spec,
    )

    return (
        f"{SYSTEM_MARKER}\n"
        f"{SYSTEM_PROMPT}\n\n"
        f"{INSTRUCTION_MARKER}\n"
        f"{user_text}\n\n"
        f"{RESPONSE_MARKER}\n"
    )


# ------------------------------------------------------------
# Prediction cleaning
# ------------------------------------------------------------

def arabic_ratio(text):
    text = safe_string(text)

    if not text:
        return 0.0

    arabic_characters = sum(
        1
        for character in text
        if "\u0600" <= character <= "\u06FF"
    )

    return arabic_characters / max(len(text), 1)


def latin_ratio(text):
    text = safe_string(text)

    if not text:
        return 0.0

    latin_characters = sum(
        1
        for character in text
        if "a" <= character.lower() <= "z"
    )

    return latin_characters / max(len(text), 1)


def strip_special_tokens(text, tokenizer=None):
    text = safe_string(text)

    special_tokens = [
        "<|endoftext|>",
        "<|im_end|>",
        "<|im_start|>",
        "<turn|>",
    ]

    if tokenizer is not None:
        for token in [
            getattr(tokenizer, "eos_token", None),
            getattr(tokenizer, "pad_token", None),
        ]:
            if token:
                special_tokens.append(token)

    for token in special_tokens:
        text = text.replace(token, "")

    return text.strip()


def postprocess_prediction(text, tokenizer=None):
    text = strip_special_tokens(
        text,
        tokenizer=tokenizer,
    )

    text = (
        text
        .replace("\r\n", "\n")
        .replace("\r", "\n")
        .strip()
    )

    split_markers = [
        "### Arabic translation:",
        "### Arabic Translation:",
        "Arabic translation:",
        "Arabic Translation:",
        "### Arabic:",
        "Arabic:",
        "الترجمة العربية:",
        "الترجمة:",
    ]

    for marker in split_markers:
        if marker in text:
            text = text.split(marker)[-1].strip()

    text = text.strip("`").strip()
    text = text.replace("###", "").strip()

    lines = [
        line.strip()
        for line in text.split("\n")
        if line.strip()
    ]

    kept_lines = []

    drop_prefixes = [
        "return only",
        "do not",
        "don't add",
        "preserve",
        "use the",
        "task:",
        "rules:",
        "metadata:",
        "previous english",
        "current english",
        "few-shot",
        "example",
        "english:",
        "arabic translation",
        "translation:",
        "target dialect",
        "speaker-to-addressee",
    ]

    for line in lines:
        lowered = line.lower().strip()

        if any(
            lowered.startswith(prefix)
            for prefix in drop_prefixes
        ):
            continue

        if (
            arabic_ratio(line) < 0.10
            and latin_ratio(line) > 0.35
        ):
            continue

        kept_lines.append(line)

    cleaned = " ".join(kept_lines).strip()

    if not cleaned and arabic_ratio(text) > 0.10:
        cleaned = " ".join([
            line
            for line in lines
            if arabic_ratio(line) > 0.10
        ]).strip()

    return cleaned


# ------------------------------------------------------------
# Model loading and generation
# ------------------------------------------------------------

def load_checkpoint_16600():
    dtype = (
        torch.bfloat16
        if (
            torch.cuda.is_available()
            and torch.cuda.is_bf16_supported()
        )
        else torch.float16
    )

    tokenizer = AutoTokenizer.from_pretrained(
        str(BASE_MODEL_DIR),
        trust_remote_code=True,
        local_files_only=True,
        use_fast=True,
    )

    tokenizer.padding_side = "left"
    tokenizer.truncation_side = "left"

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(
        str(BASE_MODEL_DIR),
        torch_dtype=dtype,
        device_map="auto",
        trust_remote_code=True,
        local_files_only=True,
        attn_implementation="sdpa",
    )

    model = PeftModel.from_pretrained(
        base_model,
        str(CHECKPOINT_PATH),
        is_trainable=False,
    )

    model.config.use_cache = bool(
        GENERATION_KWARGS["use_cache"]
    )

    model.eval()

    return model, tokenizer


def generate_batch_safe(
    model,
    tokenizer,
    row_dicts,
    spec,
):
    prompts = [
        format_generation_prompt(
            row_dict,
            spec,
        )
        for row_dict in row_dicts
    ]

    try:
        encoded = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LENGTH,
        )

        device = next(model.parameters()).device

        encoded = {
            key: value.to(device)
            for key, value in encoded.items()
        }

        padded_prompt_length = (
            encoded["input_ids"].shape[-1]
        )

        with torch.inference_mode():
            generated = model.generate(
                **encoded,
                max_new_tokens=MAX_NEW_TOKENS,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id,
                **GENERATION_KWARGS,
            )

        results = []

        for row_index in range(len(row_dicts)):
            generated_ids = generated[
                row_index,
                padded_prompt_length:,
            ]

            raw_prediction = tokenizer.decode(
                generated_ids,
                skip_special_tokens=False,
            )

            raw_prediction = strip_special_tokens(
                raw_prediction,
                tokenizer=tokenizer,
            )

            clean_prediction = postprocess_prediction(
                raw_prediction,
                tokenizer=tokenizer,
            )

            generation_error = ""

            final_prediction = (
                clean_prediction
                if clean_prediction
                else raw_prediction
            ).strip()

            if not final_prediction:
                generation_error = (
                    "empty_prediction_after_cleaning"
                )

            results.append({
                "raw_prediction": raw_prediction,
                "clean_prediction": clean_prediction,
                "prediction": final_prediction,
                "generation_error": generation_error,
            })

        return results

    except RuntimeError as error:
        error_text = str(error)

        if len(row_dicts) > 1:
            print(
                "\nRuntimeError for batch of",
                len(row_dicts),
                "rows. Splitting batch.",
            )
            print("Reason:", error_text[:500])

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            midpoint = len(row_dicts) // 2

            return (
                generate_batch_safe(
                    model,
                    tokenizer,
                    row_dicts[:midpoint],
                    spec,
                )
                + generate_batch_safe(
                    model,
                    tokenizer,
                    row_dicts[midpoint:],
                    spec,
                )
            )

        return [{
            "raw_prediction": "",
            "clean_prediction": "",
            "prediction": "",
            "generation_error": (
                f"{type(error).__name__}: "
                f"{error_text}"
            ),
        }]


# ------------------------------------------------------------
# Official scoring and submission packaging
# ------------------------------------------------------------

def score_and_package_variant(
    prediction_df,
    variant_dir,
    variant_name,
    fingerprint,
):
    prediction_df = prediction_df.copy()

    prediction_df["source_id"] = (
        prediction_df["source_id"].astype(str)
    )

    prediction_df["prediction"] = (
        prediction_df["prediction"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    expected_ids = set(
        official_dev_df["source_id"].astype(str)
    )

    actual_ids = set(
        prediction_df["source_id"].astype(str)
    )

    missing_ids = expected_ids - actual_ids
    extra_ids = actual_ids - expected_ids

    if missing_ids:
        raise RuntimeError(
            f"Cannot score incomplete variant. "
            f"Missing IDs={len(missing_ids)}"
        )

    if extra_ids:
        raise RuntimeError(
            f"Variant contains extra IDs={len(extra_ids)}"
        )

    if prediction_df["source_id"].duplicated().any():
        raise RuntimeError(
            "Duplicate source_id in predictions."
        )

    if int(
        (
            prediction_df["prediction"] == ""
        ).sum()
    ):
        raise RuntimeError(
            "Empty predictions remain."
        )

    base_columns = [
        "source_id",
        "config",
        "country",
        "conversation_id",
        "turn_order",
        "source_text",
        "reference_arabic",
    ]

    scored_df = official_dev_df[
        base_columns
    ].merge(
        prediction_df[
            [
                "source_id",
                "prediction",
                "raw_prediction",
                "clean_prediction",
                "generation_error",
            ]
        ],
        on="source_id",
        how="left",
        validate="one_to_one",
    )

    per_country_rows = []

    for country in OFFICIAL_DEV_CONFIGS:
        country_df = scored_df[
            scored_df["config"] == country
        ].copy()

        predictions = (
            country_df["prediction"]
            .astype(str)
            .tolist()
        )

        references = (
            country_df["reference_arabic"]
            .astype(str)
            .tolist()
        )

        spbleu = sacrebleu.corpus_bleu(
            predictions,
            [references],
            tokenize="flores200",
        ).score

        chrfpp = sacrebleu.corpus_chrf(
            predictions,
            [references],
            word_order=2,
        ).score

        bleu = sacrebleu.corpus_bleu(
            predictions,
            [references],
        ).score

        chrf = sacrebleu.corpus_chrf(
            predictions,
            [references],
            word_order=0,
        ).score

        per_country_rows.append({
            "country": country,
            "turns": int(len(country_df)),
            "spBLEU": float(spbleu),
            "chrF++": float(chrfpp),
            "BLEU": float(bleu),
            "chrF": float(chrf),
        })

    per_country_df = pd.DataFrame(
        per_country_rows
    )

    average_spbleu = float(
        per_country_df["spBLEU"].mean()
    )

    average_chrfpp = float(
        per_country_df["chrF++"].mean()
    )

    # Wide, leaderboard-like single row.
    official_score_row = {
        "Variant": variant_name,
        "Checkpoint": CHECKPOINT_STEP,
        "Average spBLEU (primary)": average_spbleu,
        "Average chrF++": average_chrfpp,
    }

    for row in per_country_rows:
        country = row["country"]

        official_score_row[
            f"{country} spBLEU"
        ] = row["spBLEU"]

        official_score_row[
            f"{country} chrF++"
        ] = row["chrF++"]

    official_score_df = pd.DataFrame([
        official_score_row
    ])

    per_country_metrics_path = (
        variant_dir
        / "per_country_official_metrics.csv"
    )

    official_score_path = (
        variant_dir
        / "official_leaderboard_score_row.csv"
    )

    official_metrics_json_path = (
        variant_dir
        / "official_metrics.json"
    )

    scored_predictions_path = (
        variant_dir
        / "scored_turn_predictions.csv"
    )

    per_country_df.to_csv(
        per_country_metrics_path,
        index=False,
        encoding="utf-8-sig",
    )

    official_score_df.to_csv(
        official_score_path,
        index=False,
        encoding="utf-8-sig",
    )

    scored_df.to_csv(
        scored_predictions_path,
        index=False,
        encoding="utf-8-sig",
    )

    official_metrics = {
        "variant_name": variant_name,
        "variant_fingerprint": fingerprint,
        "checkpoint_step": CHECKPOINT_STEP,
        "checkpoint_path": str(CHECKPOINT_PATH),
        "official_split": OFFICIAL_SPLIT,
        "num_countries": int(
            len(OFFICIAL_DEV_CONFIGS)
        ),
        "num_turns": int(len(scored_df)),
        "Average spBLEU (primary)": average_spbleu,
        "Average chrF++": average_chrfpp,
        "per_country": per_country_rows,
        "sacrebleu_version": sacrebleu.__version__,
        "spbleu_tokenizer": "flores200",
        "chrf_word_order": 2,
        "completed_at": time.strftime(
            "%Y-%m-%d %H:%M:%S"
        ),
    }

    with open(
        official_metrics_json_path,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            official_metrics,
            f,
            ensure_ascii=False,
            indent=2,
        )

    # --------------------------------------------------------
    # Build official predictions.jsonl
    # --------------------------------------------------------

    submission_df = scored_df.sort_values(
        [
            "config",
            "conversation_id",
            "turn_order",
        ]
    ).reset_index(drop=True)

    submission_records = []

    for (
        country,
        conversation_id,
    ), conversation_df in submission_df.groupby(
        ["config", "conversation_id"],
        sort=True,
    ):
        conversation_df = conversation_df.sort_values(
            "turn_order"
        )

        if conversation_df[
            "turn_order"
        ].duplicated().any():
            raise RuntimeError(
                "Duplicate turn_order inside "
                f"{country}/{conversation_id}"
            )

        turns = [
            {
                "turn_order": int(row["turn_order"]),
                "prediction": str(row["prediction"]),
            }
            for _, row in conversation_df.iterrows()
        ]

        submission_records.append({
            "conv_id": str(conversation_id),
            "country": str(country),
            "turns": turns,
        })

    submission_turn_count = sum(
        len(record["turns"])
        for record in submission_records
    )

    if submission_turn_count != EXPECTED_DEV_TURNS:
        raise RuntimeError(
            "Submission turn count mismatch: "
            f"{submission_turn_count}"
        )

    jsonl_path = variant_dir / "predictions.jsonl"

    zip_path = (
        variant_dir
        / "submission_predictions.zip"
    )

    with open(
        jsonl_path,
        "w",
        encoding="utf-8",
    ) as f:
        for record in submission_records:
            f.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                )
                + "\n"
            )

    with zipfile.ZipFile(
        zip_path,
        "w",
        compression=zipfile.ZIP_DEFLATED,
    ) as zip_file:
        zip_file.write(
            jsonl_path,
            arcname="predictions.jsonl",
        )

    # --------------------------------------------------------
    # Readback validation
    # --------------------------------------------------------

    readback_keys = set()
    readback_turn_count = 0

    with zipfile.ZipFile(zip_path, "r") as zip_file:
        names = zip_file.namelist()

        if names != ["predictions.jsonl"]:
            raise RuntimeError(
                "ZIP must contain only predictions.jsonl. "
                f"Found: {names}"
            )

        with zip_file.open(
            "predictions.jsonl",
            "r",
        ) as f:
            for binary_line in f:
                record = json.loads(
                    binary_line.decode("utf-8")
                )

                country = str(record["country"])
                conversation_id = str(
                    record["conv_id"]
                )

                for turn in record["turns"]:
                    readback_turn_count += 1

                    readback_keys.add((
                        country,
                        conversation_id,
                        int(turn["turn_order"]),
                    ))

    expected_official_keys = set(zip(
        official_dev_df["config"].astype(str),
        official_dev_df[
            "conversation_id"
        ].astype(str),
        official_dev_df[
            "turn_order"
        ].astype(int),
    ))

    if readback_turn_count != EXPECTED_DEV_TURNS:
        raise RuntimeError(
            "ZIP readback turn mismatch: "
            f"{readback_turn_count}"
        )

    if readback_keys != expected_official_keys:
        raise RuntimeError(
            "ZIP official-key set does not match DEV."
        )

    # --------------------------------------------------------
    # Display official-style results
    # --------------------------------------------------------

    print("\n" + "=" * 90)
    print("OFFICIAL-STYLE DEVELOPMENT RESULT")
    print("=" * 90)
    print("Variant:", variant_name)
    print("Checkpoint:", CHECKPOINT_STEP)
    print("Turns:", len(scored_df))
    print("Countries:", len(OFFICIAL_DEV_CONFIGS))
    print(
        f"Average spBLEU (primary): "
        f"{average_spbleu:.4f}"
    )
    print(
        f"Average chrF++: "
        f"{average_chrfpp:.4f}"
    )

    print("\nLeaderboard-like row:")
    display(official_score_df)

    print("\nPer-country scores:")
    display(per_country_df)

    print("\nSaved artifacts:")
    print("Turn predictions:", scored_predictions_path)
    print("Per-country metrics:", per_country_metrics_path)
    print("Official score row:", official_score_path)
    print("Metrics JSON:", official_metrics_json_path)
    print("JSONL:", jsonl_path)
    print("ZIP:", zip_path)

    print("\n✅ SUBMIT THIS ZIP:")
    print(zip_path)

    return {
        "variant_name": variant_name,
        "variant_dir": variant_dir,
        "average_spbleu": average_spbleu,
        "average_chrfpp": average_chrfpp,
        "per_country_df": per_country_df,
        "official_score_df": official_score_df,
        "jsonl_path": jsonl_path,
        "zip_path": zip_path,
        "metrics_path": official_metrics_json_path,
    }


# ------------------------------------------------------------
# Complete resumable variant runner
# ------------------------------------------------------------

def run_inference_variant(spec):
    required_spec_keys = {
        "variant_name",
        "description",
        "shot_mode",
        "use_direction",
        "use_previous_speakers",
        "use_participants",
    }

    missing_spec_keys = (
        required_spec_keys - set(spec)
    )

    if missing_spec_keys:
        raise ValueError(
            f"Variant spec missing keys: "
            f"{sorted(missing_spec_keys)}"
        )

    variant_name = spec["variant_name"]

    if not re.fullmatch(
        r"[A-Za-z0-9_.-]+",
        variant_name,
    ):
        raise ValueError(
            "variant_name may contain only letters, "
            "numbers, underscore, dash, and dot."
        )

    if spec["shot_mode"] not in {
        "none",
        "random",
        "retrieved",
    }:
        raise ValueError(
            f"Invalid shot_mode: "
            f"{spec['shot_mode']}"
        )

    variant_dir = (
        INFERENCE_VARIANTS_ROOT
        / variant_name
    )

    variant_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    fingerprint = variant_fingerprint(spec)

    manifest_path = (
        variant_dir / "variant_manifest.json"
    )

    manifest = {
        "variant_name": variant_name,
        "description": spec["description"],
        "variant_spec": spec,
        "variant_fingerprint": fingerprint,
        "checkpoint_step": CHECKPOINT_STEP,
        "checkpoint_path": str(CHECKPOINT_PATH),
        "base_model_dir": str(BASE_MODEL_DIR),
        "prompt_version": PROMPT_VERSION,
        "generation_kwargs": GENERATION_KWARGS,
        "created_at": time.strftime(
            "%Y-%m-%d %H:%M:%S"
        ),
    }

    if manifest_path.exists():
        with open(
            manifest_path,
            "r",
            encoding="utf-8",
        ) as f:
            existing_manifest = json.load(f)

        existing_fingerprint = (
            existing_manifest.get(
                "variant_fingerprint",
                "",
            )
        )

        if existing_fingerprint != fingerprint:
            raise RuntimeError(
                "Variant fingerprint mismatch.\n"
                "The existing folder was generated with "
                "different settings.\n"
                "Use a new variant_name instead of mixing "
                "predictions."
            )
    else:
        with open(
            manifest_path,
            "w",
            encoding="utf-8",
        ) as f:
            json.dump(
                manifest,
                f,
                ensure_ascii=False,
                indent=2,
            )

    print("\n" + "=" * 90)
    print("START/RESUME INFERENCE VARIANT")
    print("=" * 90)
    print("Variant:", variant_name)
    print("Description:", spec["description"])
    print("Folder:", variant_dir)
    print("Fingerprint:", fingerprint)
    print("Shot mode:", spec["shot_mode"])
    print(
        "Use direction:",
        spec["use_direction"],
    )
    print(
        "Use previous speakers:",
        spec["use_previous_speakers"],
    )
    print(
        "Use participants:",
        spec["use_participants"],
    )
    print("Checkpoint:", CHECKPOINT_PATH)

    variant_df = prepare_variant_dataframe(
        spec=spec,
        variant_dir=variant_dir,
        fingerprint=fingerprint,
    )

    prediction_path = (
        variant_dir / "turn_predictions.csv"
    )

    expected_ids = set(
        variant_df["source_id"].astype(str)
    )

    order_df = variant_df[
        ["source_id"]
    ].copy()

    order_df["eval_order"] = np.arange(
        len(order_df),
        dtype=np.int64,
    )

    def save_prediction_rows(prediction_rows_by_id):
        saved_df = pd.DataFrame(
            list(prediction_rows_by_id.values())
        )

        if len(saved_df):
            saved_df["source_id"] = (
                saved_df["source_id"].astype(str)
            )

            saved_df = saved_df.merge(
                order_df,
                on="source_id",
                how="left",
                validate="one_to_one",
            )

            saved_df = (
                saved_df
                .sort_values("eval_order")
                .drop(columns=["eval_order"])
                .reset_index(drop=True)
            )

        saved_df.to_csv(
            prediction_path,
            index=False,
            encoding="utf-8-sig",
        )

        return saved_df

    # --------------------------------------------------------
    # Resume only valid completed predictions
    # --------------------------------------------------------

    prediction_rows_by_id = {}

    if prediction_path.exists():
        existing_df = pd.read_csv(
            prediction_path
        )

        if "source_id" not in existing_df.columns:
            raise RuntimeError(
                f"Invalid prediction file: "
                f"{prediction_path}"
            )

        existing_df["source_id"] = (
            existing_df["source_id"].astype(str)
        )

        existing_df = existing_df[
            existing_df["source_id"].isin(
                expected_ids
            )
        ].copy()

        existing_df = existing_df.drop_duplicates(
            subset=["source_id"],
            keep="last",
        )

        if (
            "variant_fingerprint"
            not in existing_df.columns
        ):
            raise RuntimeError(
                "Existing prediction CSV has no "
                "variant fingerprint."
            )

        fingerprint_values = set(
            existing_df[
                "variant_fingerprint"
            ].dropna().astype(str)
        )

        if (
            fingerprint_values
            and fingerprint_values != {fingerprint}
        ):
            raise RuntimeError(
                "Existing predictions were generated "
                "with a different variant fingerprint."
            )

        valid_mask = (
            existing_df["prediction"]
            .fillna("")
            .astype(str)
            .str.strip()
            != ""
        )

        if "generation_error" in existing_df.columns:
            valid_mask &= (
                existing_df["generation_error"]
                .fillna("")
                .astype(str)
                .str.strip()
                == ""
            )

        valid_existing_df = existing_df[
            valid_mask
        ].copy()

        prediction_rows_by_id = {
            str(row["source_id"]): row
            for row in valid_existing_df.to_dict(
                "records"
            )
        }

        print(
            "Resuming valid predictions:",
            len(prediction_rows_by_id),
            "/",
            len(variant_df),
        )

        retry_count = (
            len(existing_df)
            - len(valid_existing_df)
        )

        if retry_count:
            print(
                "Rows to retry because of "
                "empty/error output:",
                retry_count,
            )

    # --------------------------------------------------------
    # Generate missing predictions
    # --------------------------------------------------------

    done_ids = set(
        prediction_rows_by_id.keys()
    )

    missing_count = (
        len(expected_ids - done_ids)
    )

    print(
        "Missing predictions:",
        missing_count,
        "/",
        len(variant_df),
    )

    model = None
    tokenizer = None

    if missing_count:
        try:
            print("\nLoading checkpoint 16600...")

            model, tokenizer = (
                load_checkpoint_16600()
            )

            pending_rows = []
            generated_since_save = 0

            for _, row in tqdm(
                variant_df.iterrows(),
                total=len(variant_df),
                desc=variant_name,
            ):
                row_dict = row.to_dict()
                source_id = str(
                    row_dict["source_id"]
                )

                if source_id in done_ids:
                    continue

                pending_rows.append(row_dict)

                if (
                    len(pending_rows)
                    < GEN_BATCH_SIZE
                ):
                    continue

                batch_results = generate_batch_safe(
                    model=model,
                    tokenizer=tokenizer,
                    row_dicts=pending_rows,
                    spec=spec,
                )

                for one_row, result in zip(
                    pending_rows,
                    batch_results,
                ):
                    source_id = str(
                        one_row["source_id"]
                    )

                    prediction_rows_by_id[
                        source_id
                    ] = {
                        "source_id": source_id,
                        "config": one_row["config"],
                        "country": one_row["country"],
                        "conversation_id": one_row[
                            "conversation_id"
                        ],
                        "turn_order": int(
                            one_row["turn_order"]
                        ),
                        "dialect": one_row["dialect"],
                        "domain": one_row["domain"],
                        "source_text": one_row[
                            "source_text"
                        ],

                        "raw_prediction": result[
                            "raw_prediction"
                        ],
                        "clean_prediction": result[
                            "clean_prediction"
                        ],
                        "prediction": result[
                            "prediction"
                        ],
                        "generation_error": result[
                            "generation_error"
                        ],

                        "variant_name": variant_name,
                        "variant_fingerprint": fingerprint,
                        "checkpoint_step": CHECKPOINT_STEP,
                        "checkpoint_path": str(
                            CHECKPOINT_PATH
                        ),
                        "decode_tag": DECODE_TAG,
                        "prompt_version": PROMPT_VERSION,
                        "shot_mode": spec["shot_mode"],
                        "use_direction": spec[
                            "use_direction"
                        ],
                        "use_previous_speakers": spec[
                            "use_previous_speakers"
                        ],
                        "use_participants": spec[
                            "use_participants"
                        ],
                        "generated_at": time.strftime(
                            "%Y-%m-%d %H:%M:%S"
                        ),
                    }

                    done_ids.add(source_id)
                    generated_since_save += 1

                pending_rows = []

                if (
                    generated_since_save
                    >= SAVE_EVERY
                ):
                    saved_df = save_prediction_rows(
                        prediction_rows_by_id
                    )

                    print(
                        "Saved:",
                        len(saved_df),
                        "/",
                        len(variant_df),
                    )

                    generated_since_save = 0

            # Final incomplete batch
            if pending_rows:
                batch_results = generate_batch_safe(
                    model=model,
                    tokenizer=tokenizer,
                    row_dicts=pending_rows,
                    spec=spec,
                )

                for one_row, result in zip(
                    pending_rows,
                    batch_results,
                ):
                    source_id = str(
                        one_row["source_id"]
                    )

                    prediction_rows_by_id[
                        source_id
                    ] = {
                        "source_id": source_id,
                        "config": one_row["config"],
                        "country": one_row["country"],
                        "conversation_id": one_row[
                            "conversation_id"
                        ],
                        "turn_order": int(
                            one_row["turn_order"]
                        ),
                        "dialect": one_row["dialect"],
                        "domain": one_row["domain"],
                        "source_text": one_row[
                            "source_text"
                        ],

                        "raw_prediction": result[
                            "raw_prediction"
                        ],
                        "clean_prediction": result[
                            "clean_prediction"
                        ],
                        "prediction": result[
                            "prediction"
                        ],
                        "generation_error": result[
                            "generation_error"
                        ],

                        "variant_name": variant_name,
                        "variant_fingerprint": fingerprint,
                        "checkpoint_step": CHECKPOINT_STEP,
                        "checkpoint_path": str(
                            CHECKPOINT_PATH
                        ),
                        "decode_tag": DECODE_TAG,
                        "prompt_version": PROMPT_VERSION,
                        "shot_mode": spec["shot_mode"],
                        "use_direction": spec[
                            "use_direction"
                        ],
                        "use_previous_speakers": spec[
                            "use_previous_speakers"
                        ],
                        "use_participants": spec[
                            "use_participants"
                        ],
                        "generated_at": time.strftime(
                            "%Y-%m-%d %H:%M:%S"
                        ),
                    }

                    done_ids.add(source_id)

        finally:
            # Save whatever was completed before cleanup.
            if prediction_rows_by_id:
                save_prediction_rows(
                    prediction_rows_by_id
                )

            if model is not None:
                del model

            if tokenizer is not None:
                del tokenizer

            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

                if hasattr(
                    torch.cuda,
                    "ipc_collect",
                ):
                    torch.cuda.ipc_collect()

    # --------------------------------------------------------
    # Final prediction validation
    # --------------------------------------------------------

    if not prediction_path.exists():
        raise RuntimeError(
            "Prediction CSV was not created."
        )

    final_prediction_df = pd.read_csv(
        prediction_path
    )

    final_prediction_df["source_id"] = (
        final_prediction_df[
            "source_id"
        ].astype(str)
    )

    final_prediction_df["prediction"] = (
        final_prediction_df["prediction"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    final_prediction_df[
        "generation_error"
    ] = (
        final_prediction_df[
            "generation_error"
        ]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    final_ids = set(
        final_prediction_df["source_id"]
    )

    missing_ids = expected_ids - final_ids
    extra_ids = final_ids - expected_ids

    duplicate_count = int(
        final_prediction_df[
            "source_id"
        ].duplicated().sum()
    )

    empty_count = int(
        (
            final_prediction_df[
                "prediction"
            ] == ""
        ).sum()
    )

    error_count = int(
        (
            final_prediction_df[
                "generation_error"
            ] != ""
        ).sum()
    )

    print("\n==============================")
    print("Final prediction validation")
    print("==============================")
    print("Rows:", len(final_prediction_df))
    print("Expected:", len(variant_df))
    print("Missing:", len(missing_ids))
    print("Extra:", len(extra_ids))
    print("Duplicates:", duplicate_count)
    print("Empty:", empty_count)
    print("Generation errors:", error_count)

    if (
        missing_ids
        or extra_ids
        or duplicate_count
        or empty_count
        or error_count
        or len(final_prediction_df)
        != len(variant_df)
    ):
        if error_count:
            print("\nFirst generation errors:")
            display(
                final_prediction_df[
                    final_prediction_df[
                        "generation_error"
                    ] != ""
                ][
                    [
                        "source_id",
                        "source_text",
                        "generation_error",
                    ]
                ].head(20)
            )

        raise RuntimeError(
            "Variant is incomplete. Re-run the same "
            "cell to retry missing/error rows."
        )

    # --------------------------------------------------------
    # Score and package
    # --------------------------------------------------------

    return score_and_package_variant(
        prediction_df=final_prediction_df,
        variant_dir=variant_dir,
        variant_name=variant_name,
        fingerprint=fingerprint,
    )


print("✅ Shared inference-variant engine is ready.")
print("Variant root:", INFERENCE_VARIANTS_ROOT)

✅ Shared inference-variant engine is ready.
Variant root: /home/mabdallah/alexandriax_mt_14d/inference_variants


### **Control: reproduce the previous official inference behavior**

In [5]:
# ============================================================
# Cell 4 — Variant 00: Previous official-inference control
#
# Purpose:
#   Reproduce the behavior of the previous official DEV
#   generation as closely as possible inside the new pipeline.
#
# Uses:
#   - checkpoint 16600
#   - zero shots
#   - no current gender direction
#   - previous English text without speaker labels
#   - participants included
#
# Complete workflow:
#   predict -> save/resume -> score -> display -> JSONL -> ZIP
# ============================================================

VARIANT_SPEC = {
    "variant_name": "00_previous_official_control",
    "description": (
        "Checkpoint 16600 control reproducing the previous "
        "official inference behavior: zero-shot, no current "
        "gender direction, no previous speaker labels, with "
        "participants metadata."
    ),
    "shot_mode": "none",
    "use_direction": False,
    "use_previous_speakers": False,
    "use_participants": True,
}

variant_00_result = run_inference_variant(
    VARIANT_SPEC
)


START/RESUME INFERENCE VARIANT
Variant: 00_previous_official_control
Description: Checkpoint 16600 control reproducing the previous official inference behavior: zero-shot, no current gender direction, no previous speaker labels, with participants metadata.
Folder: /home/mabdallah/alexandriax_mt_14d/inference_variants/00_previous_official_control
Fingerprint: 8c6a7a98cb99f3495da22151ff62d8643b3985055256647124731aa7ca4a786d
Shot mode: none
Use direction: False
Use previous speakers: False
Use participants: True
Checkpoint: /home/mabdallah/alexandriax_mt_14d/runs/nilechat3b_all14/nilechat3b_alexandria_all14_context3_complete2shot_all_group_r16_alpha32_3epochs_beam4_nonquant_server5090_v1/checkpoint-16600
Missing predictions: 12250 / 12250

Loading checkpoint 16600...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/435 [00:00<?, ?it/s]

00_previous_official_control:   0%|          | 0/12250 [00:00<?, ?it/s]

Saved: 50 / 12250
Saved: 100 / 12250
Saved: 150 / 12250
Saved: 200 / 12250
Saved: 250 / 12250
Saved: 300 / 12250
Saved: 350 / 12250
Saved: 400 / 12250
Saved: 450 / 12250
Saved: 500 / 12250
Saved: 550 / 12250
Saved: 600 / 12250
Saved: 650 / 12250
Saved: 700 / 12250
Saved: 750 / 12250
Saved: 800 / 12250
Saved: 850 / 12250
Saved: 900 / 12250
Saved: 950 / 12250
Saved: 1000 / 12250
Saved: 1050 / 12250
Saved: 1100 / 12250
Saved: 1150 / 12250
Saved: 1200 / 12250
Saved: 1250 / 12250
Saved: 1300 / 12250
Saved: 1350 / 12250
Saved: 1400 / 12250
Saved: 1450 / 12250
Saved: 1500 / 12250
Saved: 1550 / 12250
Saved: 1600 / 12250
Saved: 1650 / 12250
Saved: 1700 / 12250
Saved: 1750 / 12250
Saved: 1800 / 12250
Saved: 1850 / 12250
Saved: 1900 / 12250
Saved: 1950 / 12250
Saved: 2000 / 12250
Saved: 2050 / 12250
Saved: 2100 / 12250
Saved: 2150 / 12250
Saved: 2200 / 12250
Saved: 2250 / 12250
Saved: 2300 / 12250
Saved: 2350 / 12250
Saved: 2400 / 12250
Saved: 2450 / 12250
Saved: 2500 / 12250
Saved: 2550 / 12250


,Variant,Checkpoint,Average spBLEU (primary),Average chrF++,EG spBLEU,EG chrF++,JO spBLEU,JO chrF++,LB spBLEU,LB chrF++,...,PS spBLEU,PS chrF++,SA spBLEU,SA chrF++,SY spBLEU,SY chrF++,TN spBLEU,TN chrF++,YE spBLEU,YE chrF++
0,00_previous_official_control,16600,30.165227,45.079987,32.255421,46.31209,34.342613,48.784156,30.748908,45.328505,...,32.449844,47.112136,32.342947,47.477985,39.827606,53.851716,28.509805,43.322662,26.522481,42.380197



Per-country scores:


,country,turns,spBLEU,chrF++,BLEU,chrF
0,EG,1113,32.255421,46.312090,18.773911,49.389169
1,JO,1113,34.342613,48.784156,18.977230,52.293360
2,LB,1118,30.748908,45.328505,19.056565,48.111051
3,MA,1110,22.819213,38.620692,12.109090,41.451815
4,MR,1114,17.114205,33.656454,7.068855,37.781869
5,OM,1109,34.884460,49.033257,19.709179,52.477283
6,PS,1110,32.449844,47.112136,20.175189,50.384181
7,SA,1110,32.342947,47.477985,17.830254,51.528051
8,SY,1119,39.827606,53.851716,26.498481,56.847388
9,TN,1116,28.509805,43.322662,16.643198,45.964899



Saved artifacts:
Turn predictions: /home/mabdallah/alexandriax_mt_14d/inference_variants/00_previous_official_control/scored_turn_predictions.csv
Per-country metrics: /home/mabdallah/alexandriax_mt_14d/inference_variants/00_previous_official_control/per_country_official_metrics.csv
Official score row: /home/mabdallah/alexandriax_mt_14d/inference_variants/00_previous_official_control/official_leaderboard_score_row.csv
Metrics JSON: /home/mabdallah/alexandriax_mt_14d/inference_variants/00_previous_official_control/official_metrics.json
JSONL: /home/mabdallah/alexandriax_mt_14d/inference_variants/00_previous_official_control/predictions.jsonl
ZIP: /home/mabdallah/alexandriax_mt_14d/inference_variants/00_previous_official_control/submission_predictions.zip

✅ SUBMIT THIS ZIP:
/home/mabdallah/alexandriax_mt_14d/inference_variants/00_previous_official_control/submission_predictions.zip


### **Exact training-parity inference**

In [6]:
# ============================================================
# Cell 5 — Variant 01: Exact training-parity inference
#
# Purpose:
#   Make checkpoint-16600 inference match the prompt structure
#   it saw during training.
#
# Uses:
#   - two deterministic random shots
#   - same-country/same-domain preference
#   - current turn gender direction
#   - previous speaker labels
#   - participants disabled because training persona was empty
#
# Complete workflow:
#   predict -> save/resume -> score -> display -> JSONL -> ZIP
# ============================================================

VARIANT_SPEC = {
    "variant_name": "01_exact_training_parity",
    "description": (
        "Checkpoint 16600 with exact training-style inference: "     
        "two deterministic random shots, current gender "
        "direction, previous speaker labels, and no participants "
        "metadata."
    ),
    "shot_mode": "random",
    "use_direction": True,
    "use_previous_speakers": True,
    "use_participants": False,
}

variant_01_result = run_inference_variant(
    VARIANT_SPEC
)


START/RESUME INFERENCE VARIANT
Variant: 01_exact_training_parity
Description: Checkpoint 16600 with exact training-style inference: two deterministic random shots, current gender direction, previous speaker labels, and no participants metadata.
Folder: /home/mabdallah/alexandriax_mt_14d/inference_variants/01_exact_training_parity
Fingerprint: 8ad049bca9e46983fad12d5a3a7fe89ae12d0b5df422bda5b3b33476554134f1
Shot mode: random
Use direction: True
Use previous speakers: True
Use participants: False
Checkpoint: /home/mabdallah/alexandriax_mt_14d/runs/nilechat3b_all14/nilechat3b_alexandria_all14_context3_complete2shot_all_group_r16_alpha32_3epochs_beam4_nonquant_server5090_v1/checkpoint-16600
Selecting deterministic random shots...


random two-shot selection:   0%|          | 0/12250 [00:00<?, ?it/s]

Saved shot selections: /home/mabdallah/alexandriax_mt_14d/inference_variants/01_exact_training_parity/selected_few_shots.pkl
Missing predictions: 12250 / 12250

Loading checkpoint 16600...


Loading weights:   0%|          | 0/435 [00:00<?, ?it/s]

01_exact_training_parity:   0%|          | 0/12250 [00:00<?, ?it/s]

Saved: 50 / 12250
Saved: 100 / 12250
Saved: 150 / 12250
Saved: 200 / 12250
Saved: 250 / 12250
Saved: 300 / 12250
Saved: 350 / 12250
Saved: 400 / 12250
Saved: 450 / 12250
Saved: 500 / 12250
Saved: 550 / 12250
Saved: 600 / 12250
Saved: 650 / 12250
Saved: 700 / 12250
Saved: 750 / 12250
Saved: 800 / 12250
Saved: 850 / 12250
Saved: 900 / 12250
Saved: 950 / 12250
Saved: 1000 / 12250
Saved: 1050 / 12250
Saved: 1100 / 12250
Saved: 1150 / 12250
Saved: 1200 / 12250
Saved: 1250 / 12250
Saved: 1300 / 12250
Saved: 1350 / 12250
Saved: 1400 / 12250
Saved: 1450 / 12250
Saved: 1500 / 12250
Saved: 1550 / 12250
Saved: 1600 / 12250
Saved: 1650 / 12250
Saved: 1700 / 12250
Saved: 1750 / 12250
Saved: 1800 / 12250
Saved: 1850 / 12250
Saved: 1900 / 12250
Saved: 1950 / 12250
Saved: 2000 / 12250
Saved: 2050 / 12250
Saved: 2100 / 12250
Saved: 2150 / 12250
Saved: 2200 / 12250
Saved: 2250 / 12250
Saved: 2300 / 12250
Saved: 2350 / 12250
Saved: 2400 / 12250
Saved: 2450 / 12250
Saved: 2500 / 12250
Saved: 2550 / 12250


,Variant,Checkpoint,Average spBLEU (primary),Average chrF++,EG spBLEU,EG chrF++,JO spBLEU,JO chrF++,LB spBLEU,LB chrF++,...,PS spBLEU,PS chrF++,SA spBLEU,SA chrF++,SY spBLEU,SY chrF++,TN spBLEU,TN chrF++,YE spBLEU,YE chrF++
0,01_exact_training_parity,16600,30.565838,45.381164,32.858879,46.987054,35.008177,49.105473,31.281904,45.583987,...,33.19046,47.623884,32.84539,47.791144,39.887929,53.908797,28.291619,43.387183,26.822309,42.532265



Per-country scores:


,country,turns,spBLEU,chrF++,BLEU,chrF
0,EG,1113,32.858879,46.987054,19.532844,49.983208
1,JO,1113,35.008177,49.105473,19.561179,52.597341
2,LB,1118,31.281904,45.583987,19.571933,48.351325
3,MA,1110,23.190544,39.020704,12.688076,41.828538
4,MR,1114,16.974170,33.640719,6.907085,37.745422
5,OM,1109,35.872836,49.611596,20.438578,53.034060
6,PS,1110,33.190460,47.623884,20.580466,50.886702
7,SA,1110,32.845390,47.791144,18.314689,51.798796
8,SY,1119,39.887929,53.908797,26.419081,56.843604
9,TN,1116,28.291619,43.387183,16.318695,46.023716



Saved artifacts:
Turn predictions: /home/mabdallah/alexandriax_mt_14d/inference_variants/01_exact_training_parity/scored_turn_predictions.csv
Per-country metrics: /home/mabdallah/alexandriax_mt_14d/inference_variants/01_exact_training_parity/per_country_official_metrics.csv
Official score row: /home/mabdallah/alexandriax_mt_14d/inference_variants/01_exact_training_parity/official_leaderboard_score_row.csv
Metrics JSON: /home/mabdallah/alexandriax_mt_14d/inference_variants/01_exact_training_parity/official_metrics.json
JSONL: /home/mabdallah/alexandriax_mt_14d/inference_variants/01_exact_training_parity/predictions.jsonl
ZIP: /home/mabdallah/alexandriax_mt_14d/inference_variants/01_exact_training_parity/submission_predictions.zip

✅ SUBMIT THIS ZIP:
/home/mabdallah/alexandriax_mt_14d/inference_variants/01_exact_training_parity/submission_predictions.zip


### **Restored metadata without few-shots**

In [7]:
# ============================================================
# Cell 6 — Variant 02: Restored metadata without few-shots
#
# Purpose:
#   Isolate the value of the missing conversational metadata.
#
# Uses:
#   - zero shots
#   - current turn gender direction
#   - previous speaker labels
#   - participants disabled
#
# Comparison:
#   Variant 02 vs Variant 00 shows the approximate effect of
#   restoring direction/speaker structure without adding shots.
#
# Complete workflow:
#   predict -> save/resume -> score -> display -> JSONL -> ZIP
# ============================================================

VARIANT_SPEC = {
    "variant_name": "02_metadata_no_shots",
    "description": (
        "Checkpoint 16600 with restored current gender "
        "direction and previous speaker labels, but without "
        "few-shot demonstrations and without participants."
    ),
    "shot_mode": "none",
    "use_direction": True,
    "use_previous_speakers": True,
    "use_participants": False,
}

variant_02_result = run_inference_variant(
    VARIANT_SPEC
)


START/RESUME INFERENCE VARIANT
Variant: 02_metadata_no_shots
Description: Checkpoint 16600 with restored current gender direction and previous speaker labels, but without few-shot demonstrations and without participants.
Folder: /home/mabdallah/alexandriax_mt_14d/inference_variants/02_metadata_no_shots
Fingerprint: 0118de2956c6a3768a77d04ff579024bd71ecb6b12e661b1c82b6a28ab4421fe
Shot mode: none
Use direction: True
Use previous speakers: True
Use participants: False
Checkpoint: /home/mabdallah/alexandriax_mt_14d/runs/nilechat3b_all14/nilechat3b_alexandria_all14_context3_complete2shot_all_group_r16_alpha32_3epochs_beam4_nonquant_server5090_v1/checkpoint-16600
Missing predictions: 12250 / 12250

Loading checkpoint 16600...


Loading weights:   0%|          | 0/435 [00:00<?, ?it/s]

02_metadata_no_shots:   0%|          | 0/12250 [00:00<?, ?it/s]

Saved: 50 / 12250
Saved: 100 / 12250
Saved: 150 / 12250
Saved: 200 / 12250
Saved: 250 / 12250
Saved: 300 / 12250
Saved: 350 / 12250
Saved: 400 / 12250
Saved: 450 / 12250
Saved: 500 / 12250
Saved: 550 / 12250
Saved: 600 / 12250
Saved: 650 / 12250
Saved: 700 / 12250
Saved: 750 / 12250
Saved: 800 / 12250
Saved: 850 / 12250
Saved: 900 / 12250
Saved: 950 / 12250
Saved: 1000 / 12250
Saved: 1050 / 12250
Saved: 1100 / 12250
Saved: 1150 / 12250
Saved: 1200 / 12250
Saved: 1250 / 12250
Saved: 1300 / 12250
Saved: 1350 / 12250
Saved: 1400 / 12250
Saved: 1450 / 12250
Saved: 1500 / 12250
Saved: 1550 / 12250
Saved: 1600 / 12250
Saved: 1650 / 12250
Saved: 1700 / 12250
Saved: 1750 / 12250
Saved: 1800 / 12250
Saved: 1850 / 12250
Saved: 1900 / 12250
Saved: 1950 / 12250
Saved: 2000 / 12250
Saved: 2050 / 12250
Saved: 2100 / 12250
Saved: 2150 / 12250
Saved: 2200 / 12250
Saved: 2250 / 12250
Saved: 2300 / 12250
Saved: 2350 / 12250
Saved: 2400 / 12250
Saved: 2450 / 12250
Saved: 2500 / 12250
Saved: 2550 / 12250


,Variant,Checkpoint,Average spBLEU (primary),Average chrF++,EG spBLEU,EG chrF++,JO spBLEU,JO chrF++,LB spBLEU,LB chrF++,...,PS spBLEU,PS chrF++,SA spBLEU,SA chrF++,SY spBLEU,SY chrF++,TN spBLEU,TN chrF++,YE spBLEU,YE chrF++
0,02_metadata_no_shots,16600,30.555001,45.341404,32.64431,46.68937,35.138322,49.313708,31.407703,45.734606,...,32.976702,47.338623,32.79623,47.713583,39.731411,53.932363,28.177898,43.113695,26.673491,42.433708



Per-country scores:


,country,turns,spBLEU,chrF++,BLEU,chrF
0,EG,1113,32.644310,46.689370,19.381163,49.701625
1,JO,1113,35.138322,49.313708,19.853148,52.812944
2,LB,1118,31.407703,45.734606,19.574602,48.472582
3,MA,1110,23.485338,39.086157,12.806291,41.916167
4,MR,1114,17.156642,33.727633,7.232233,37.838995
5,OM,1109,35.916965,49.672000,20.534462,53.065870
6,PS,1110,32.976702,47.338623,20.493340,50.584540
7,SA,1110,32.796230,47.713583,18.217677,51.721894
8,SY,1119,39.731411,53.932363,26.357170,56.948807
9,TN,1116,28.177898,43.113695,16.335369,45.772514



Saved artifacts:
Turn predictions: /home/mabdallah/alexandriax_mt_14d/inference_variants/02_metadata_no_shots/scored_turn_predictions.csv
Per-country metrics: /home/mabdallah/alexandriax_mt_14d/inference_variants/02_metadata_no_shots/per_country_official_metrics.csv
Official score row: /home/mabdallah/alexandriax_mt_14d/inference_variants/02_metadata_no_shots/official_leaderboard_score_row.csv
Metrics JSON: /home/mabdallah/alexandriax_mt_14d/inference_variants/02_metadata_no_shots/official_metrics.json
JSONL: /home/mabdallah/alexandriax_mt_14d/inference_variants/02_metadata_no_shots/predictions.jsonl
ZIP: /home/mabdallah/alexandriax_mt_14d/inference_variants/02_metadata_no_shots/submission_predictions.zip

✅ SUBMIT THIS ZIP:
/home/mabdallah/alexandriax_mt_14d/inference_variants/02_metadata_no_shots/submission_predictions.zip


### **Semantically retrieved two-shot inference**

In [8]:
# ============================================================
# Cell 7 — Variant 03: Semantically retrieved two-shot inference
#
# Purpose:
#   Replace random shots with relevant training demonstrations.
#
# Retrieval prioritizes:
#   - same country and domain
#   - source semantic similarity
#   - same subdialect
#   - same gender direction
#   - same speaker role
#   - examples from different training conversations
#
# Uses:
#   - two retrieved shots
#   - current gender direction
#   - previous speaker labels
#   - participants disabled
#
# First execution builds and caches MiniLM embeddings.
# Later executions reuse the cached embeddings and shot choices.
#
# Complete workflow:
#   retrieve -> predict -> save/resume -> score
#   -> display -> JSONL -> ZIP
# ============================================================

VARIANT_SPEC = {
    "variant_name": "03_retrieved_two_shot",
    "description": (
        "Checkpoint 16600 with two semantically retrieved "
        "same-country demonstrations, current gender direction, "
        "previous speaker labels, and no participants metadata."
    ),
    "shot_mode": "retrieved",
    "use_direction": True,
    "use_previous_speakers": True,
    "use_participants": False,
}

variant_03_result = run_inference_variant(
    VARIANT_SPEC
)


START/RESUME INFERENCE VARIANT
Variant: 03_retrieved_two_shot
Description: Checkpoint 16600 with two semantically retrieved same-country demonstrations, current gender direction, previous speaker labels, and no participants metadata.
Folder: /home/mabdallah/alexandriax_mt_14d/inference_variants/03_retrieved_two_shot
Fingerprint: 7a2cb72d7aab1a00e7ee95c3995c159785433c7fe6b74f8782fc22aea968618e
Shot mode: retrieved
Use direction: True
Use previous speakers: True
Use participants: False
Checkpoint: /home/mabdallah/alexandriax_mt_14d/runs/nilechat3b_all14/nilechat3b_alexandria_all14_context3_complete2shot_all_group_r16_alpha32_3epochs_beam4_nonquant_server5090_v1/checkpoint-16600
Building semantic retrieval embeddings...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Saved: 50 / 12250
Saved: 100 / 12250
Saved: 150 / 12250
Saved: 200 / 12250
Saved: 250 / 12250
Saved: 300 / 12250
Saved: 350 / 12250
Saved: 400 / 12250
Saved: 450 / 12250
Saved: 500 / 12250
Saved: 550 / 12250
Saved: 600 / 12250
Saved: 650 / 12250
Saved: 700 / 12250
Saved: 750 / 12250
Saved: 800 / 12250
Saved: 850 / 12250
Saved: 900 / 12250
Saved: 950 / 12250
Saved: 1000 / 12250
Saved: 1050 / 12250
Saved: 1100 / 12250
Saved: 1150 / 12250
Saved: 1200 / 12250
Saved: 1250 / 12250
Saved: 1300 / 12250
Saved: 1350 / 12250
Saved: 1400 / 12250
Saved: 1450 / 12250
Saved: 1500 / 12250
Saved: 1550 / 12250
Saved: 1600 / 12250
Saved: 1650 / 12250
Saved: 1700 / 12250
Saved: 1750 / 12250
Saved: 1800 / 12250
Saved: 1850 / 12250
Saved: 1900 / 12250
Saved: 1950 / 12250
Saved: 2000 / 12250
Saved: 2050 / 12250
Saved: 2100 / 12250
Saved: 2150 / 12250
Saved: 2200 / 12250
Saved: 2250 / 12250
Saved: 2300 / 12250
Saved: 2350 / 12250
Saved: 2400 / 12250
Saved: 2450 / 12250
Saved: 2500 / 12250
Saved: 2550 / 12250


,Variant,Checkpoint,Average spBLEU (primary),Average chrF++,EG spBLEU,EG chrF++,JO spBLEU,JO chrF++,LB spBLEU,LB chrF++,...,PS spBLEU,PS chrF++,SA spBLEU,SA chrF++,SY spBLEU,SY chrF++,TN spBLEU,TN chrF++,YE spBLEU,YE chrF++
0,03_retrieved_two_shot,16600,30.693517,45.491812,32.717798,46.715794,35.301943,49.377895,31.467859,45.831343,...,32.817707,47.410689,33.012267,47.956335,39.961182,53.979983,28.348143,43.414589,26.998002,42.721976



Per-country scores:


,country,turns,spBLEU,chrF++,BLEU,chrF
0,EG,1113,32.717798,46.715794,19.303487,49.716994
1,JO,1113,35.301943,49.377895,19.819243,52.901916
2,LB,1118,31.467859,45.831343,19.797523,48.571026
3,MA,1110,23.504150,39.343965,12.673824,42.204498
4,MR,1114,17.367289,33.866626,7.179331,38.002673
5,OM,1109,36.132344,49.790737,20.826762,53.214611
6,PS,1110,32.817707,47.410689,20.327749,50.730572
7,SA,1110,33.012267,47.956335,18.305254,51.988206
8,SY,1119,39.961182,53.979983,26.574044,56.951978
9,TN,1116,28.348143,43.414589,16.462031,46.038025



Saved artifacts:
Turn predictions: /home/mabdallah/alexandriax_mt_14d/inference_variants/03_retrieved_two_shot/scored_turn_predictions.csv
Per-country metrics: /home/mabdallah/alexandriax_mt_14d/inference_variants/03_retrieved_two_shot/per_country_official_metrics.csv
Official score row: /home/mabdallah/alexandriax_mt_14d/inference_variants/03_retrieved_two_shot/official_leaderboard_score_row.csv
Metrics JSON: /home/mabdallah/alexandriax_mt_14d/inference_variants/03_retrieved_two_shot/official_metrics.json
JSONL: /home/mabdallah/alexandriax_mt_14d/inference_variants/03_retrieved_two_shot/predictions.jsonl
ZIP: /home/mabdallah/alexandriax_mt_14d/inference_variants/03_retrieved_two_shot/submission_predictions.zip

✅ SUBMIT THIS ZIP:
/home/mabdallah/alexandriax_mt_14d/inference_variants/03_retrieved_two_shot/submission_predictions.zip


### **Participants ablation on top of training parity**

In [9]:
# ============================================================
# Cell 8 — Variant 04: Training parity plus participants
#
# Purpose:
#   Test whether adding the released participants/roles field
#   helps or harms checkpoint 16600.
#
# Uses the same settings as Variant 01, except:
#   - participants metadata is added
#
# Direct comparison:
#   Variant 04 vs Variant 01 = effect of participants
#
# Complete workflow:
#   predict -> save/resume -> score -> display -> JSONL -> ZIP
# ============================================================

VARIANT_SPEC = {
    "variant_name": "04_training_parity_with_participants",
    "description": (
        "Checkpoint 16600 with two deterministic random shots, "
        "current gender direction, previous speaker labels, and "
        "participants metadata added as a separate ablation."
    ),
    "shot_mode": "random",
    "use_direction": True,
    "use_previous_speakers": True,
    "use_participants": True,
}

variant_04_result = run_inference_variant(
    VARIANT_SPEC
)


START/RESUME INFERENCE VARIANT
Variant: 04_training_parity_with_participants
Description: Checkpoint 16600 with two deterministic random shots, current gender direction, previous speaker labels, and participants metadata added as a separate ablation.
Folder: /home/mabdallah/alexandriax_mt_14d/inference_variants/04_training_parity_with_participants
Fingerprint: 0abd345151529d57d50daacc698e70d978df619d2e167920b042050e9d8da2c4
Shot mode: random
Use direction: True
Use previous speakers: True
Use participants: True
Checkpoint: /home/mabdallah/alexandriax_mt_14d/runs/nilechat3b_all14/nilechat3b_alexandria_all14_context3_complete2shot_all_group_r16_alpha32_3epochs_beam4_nonquant_server5090_v1/checkpoint-16600
Selecting deterministic random shots...


random two-shot selection:   0%|          | 0/12250 [00:00<?, ?it/s]

Saved shot selections: /home/mabdallah/alexandriax_mt_14d/inference_variants/04_training_parity_with_participants/selected_few_shots.pkl
Missing predictions: 12250 / 12250

Loading checkpoint 16600...


Loading weights:   0%|          | 0/435 [00:00<?, ?it/s]

04_training_parity_with_participants:   0%|          | 0/12250 [00:00<?, ?it/s]

Saved: 50 / 12250
Saved: 100 / 12250
Saved: 150 / 12250
Saved: 200 / 12250
Saved: 250 / 12250
Saved: 300 / 12250
Saved: 350 / 12250
Saved: 400 / 12250
Saved: 450 / 12250
Saved: 500 / 12250
Saved: 550 / 12250
Saved: 600 / 12250
Saved: 650 / 12250
Saved: 700 / 12250
Saved: 750 / 12250
Saved: 800 / 12250
Saved: 850 / 12250
Saved: 900 / 12250
Saved: 950 / 12250
Saved: 1000 / 12250
Saved: 1050 / 12250
Saved: 1100 / 12250
Saved: 1150 / 12250
Saved: 1200 / 12250
Saved: 1250 / 12250
Saved: 1300 / 12250
Saved: 1350 / 12250
Saved: 1400 / 12250
Saved: 1450 / 12250
Saved: 1500 / 12250
Saved: 1550 / 12250
Saved: 1600 / 12250
Saved: 1650 / 12250
Saved: 1700 / 12250
Saved: 1750 / 12250
Saved: 1800 / 12250
Saved: 1850 / 12250
Saved: 1900 / 12250
Saved: 1950 / 12250
Saved: 2000 / 12250
Saved: 2050 / 12250
Saved: 2100 / 12250
Saved: 2150 / 12250
Saved: 2200 / 12250
Saved: 2250 / 12250
Saved: 2300 / 12250
Saved: 2350 / 12250
Saved: 2400 / 12250
Saved: 2450 / 12250
Saved: 2500 / 12250
Saved: 2550 / 12250


,Variant,Checkpoint,Average spBLEU (primary),Average chrF++,EG spBLEU,EG chrF++,JO spBLEU,JO chrF++,LB spBLEU,LB chrF++,...,PS spBLEU,PS chrF++,SA spBLEU,SA chrF++,SY spBLEU,SY chrF++,TN spBLEU,TN chrF++,YE spBLEU,YE chrF++
0,04_training_parity_with_participants,16600,30.5545,45.409125,32.693573,46.924482,34.928057,49.197996,31.408079,45.732309,...,33.081268,47.616727,32.990052,47.987532,39.897545,53.966974,28.113856,43.181388,26.756003,42.55443



Per-country scores:


,country,turns,spBLEU,chrF++,BLEU,chrF
0,EG,1113,32.693573,46.924482,19.480567,49.922430
1,JO,1113,34.928057,49.197996,19.506406,52.713231
2,LB,1118,31.408079,45.732309,19.754081,48.450781
3,MA,1110,23.317194,39.090087,12.637778,41.916617
4,MR,1114,17.092337,33.697051,7.074517,37.806110
5,OM,1109,35.821542,49.551395,20.558523,52.938736
6,PS,1110,33.081268,47.616727,20.483020,50.912377
7,SA,1110,32.990052,47.987532,18.409675,52.008758
8,SY,1119,39.897545,53.966974,26.557041,56.929438
9,TN,1116,28.113856,43.181388,16.278918,45.825426



Saved artifacts:
Turn predictions: /home/mabdallah/alexandriax_mt_14d/inference_variants/04_training_parity_with_participants/scored_turn_predictions.csv
Per-country metrics: /home/mabdallah/alexandriax_mt_14d/inference_variants/04_training_parity_with_participants/per_country_official_metrics.csv
Official score row: /home/mabdallah/alexandriax_mt_14d/inference_variants/04_training_parity_with_participants/official_leaderboard_score_row.csv
Metrics JSON: /home/mabdallah/alexandriax_mt_14d/inference_variants/04_training_parity_with_participants/official_metrics.json
JSONL: /home/mabdallah/alexandriax_mt_14d/inference_variants/04_training_parity_with_participants/predictions.jsonl
ZIP: /home/mabdallah/alexandriax_mt_14d/inference_variants/04_training_parity_with_participants/submission_predictions.zip

✅ SUBMIT THIS ZIP:
/home/mabdallah/alexandriax_mt_14d/inference_variants/04_training_parity_with_participants/submission_predictions.zip


### **Retrieved two-shot inference with participants**

In [11]:
# ============================================================
# Variant 05 — Retrieved two-shot inference with participants
#
# Purpose:
#   Test whether participants/role metadata improves the strong
#   retrieved-two-shot strategy from Variant 03.
#
# Combines:
#   - two semantically retrieved demonstrations
#   - current speaker-to-addressee gender direction
#   - previous speaker labels
#   - participants/roles metadata
#
# Comparison:
#   Variant 05 vs Variant 03 isolates the participants effect
#   under retrieved-shot inference.
#
# Complete workflow:
#   select/cache shots -> predict -> save/resume -> score
#   -> display official metrics -> JSONL -> submission ZIP
# ============================================================

VARIANT_SPEC = {
    "variant_name": "05_retrieved_two_shot_with_participants",
    "description": (
        "Checkpoint 16600 with two semantically retrieved "
        "same-country demonstrations, current speaker-to-"
        "addressee gender direction, previous speaker labels, "
        "and participants/roles metadata."
    ),
    "shot_mode": "retrieved",
    "use_direction": True,
    "use_previous_speakers": True,
    "use_participants": True,
}

variant_05_result = run_inference_variant(
    VARIANT_SPEC
)


START/RESUME INFERENCE VARIANT
Variant: 05_retrieved_two_shot_with_participants
Description: Checkpoint 16600 with two semantically retrieved same-country demonstrations, current speaker-to-addressee gender direction, previous speaker labels, and participants/roles metadata.
Folder: /home/mabdallah/alexandriax_mt_14d/inference_variants/05_retrieved_two_shot_with_participants
Fingerprint: 807f00ece5631eb918a4c5641c11cdfa01d255d6741edae22c4eefcd9f798e95
Shot mode: retrieved
Use direction: True
Use previous speakers: True
Use participants: True
Checkpoint: /home/mabdallah/alexandriax_mt_14d/runs/nilechat3b_all14/nilechat3b_alexandria_all14_context3_complete2shot_all_group_r16_alpha32_3epochs_beam4_nonquant_server5090_v1/checkpoint-16600
Loading cached semantic embeddings...
Selecting semantically retrieved shots...


retrieved two-shot selection:   0%|          | 0/12250 [00:00<?, ?it/s]

Saved shot selections: /home/mabdallah/alexandriax_mt_14d/inference_variants/05_retrieved_two_shot_with_participants/selected_few_shots.pkl
Missing predictions: 12250 / 12250

Loading checkpoint 16600...


Loading weights:   0%|          | 0/435 [00:00<?, ?it/s]

05_retrieved_two_shot_with_participants:   0%|          | 0/12250 [00:00<?, ?it/s]

Saved: 50 / 12250
Saved: 100 / 12250
Saved: 150 / 12250
Saved: 200 / 12250
Saved: 250 / 12250
Saved: 300 / 12250
Saved: 350 / 12250
Saved: 400 / 12250
Saved: 450 / 12250
Saved: 500 / 12250
Saved: 550 / 12250
Saved: 600 / 12250
Saved: 650 / 12250
Saved: 700 / 12250
Saved: 750 / 12250
Saved: 800 / 12250
Saved: 850 / 12250
Saved: 900 / 12250
Saved: 950 / 12250
Saved: 1000 / 12250
Saved: 1050 / 12250
Saved: 1100 / 12250
Saved: 1150 / 12250
Saved: 1200 / 12250
Saved: 1250 / 12250
Saved: 1300 / 12250
Saved: 1350 / 12250
Saved: 1400 / 12250
Saved: 1450 / 12250
Saved: 1500 / 12250
Saved: 1550 / 12250
Saved: 1600 / 12250
Saved: 1650 / 12250
Saved: 1700 / 12250
Saved: 1750 / 12250
Saved: 1800 / 12250
Saved: 1850 / 12250
Saved: 1900 / 12250
Saved: 1950 / 12250
Saved: 2000 / 12250
Saved: 2050 / 12250
Saved: 2100 / 12250
Saved: 2150 / 12250
Saved: 2200 / 12250
Saved: 2250 / 12250
Saved: 2300 / 12250
Saved: 2350 / 12250
Saved: 2400 / 12250
Saved: 2450 / 12250
Saved: 2500 / 12250
Saved: 2550 / 12250


,Variant,Checkpoint,Average spBLEU (primary),Average chrF++,EG spBLEU,EG chrF++,JO spBLEU,JO chrF++,LB spBLEU,LB chrF++,...,PS spBLEU,PS chrF++,SA spBLEU,SA chrF++,SY spBLEU,SY chrF++,TN spBLEU,TN chrF++,YE spBLEU,YE chrF++
0,05_retrieved_two_shot_with_participants,16600,30.695985,45.484384,32.903189,46.892198,35.092749,49.22537,31.444152,45.774384,...,33.00947,47.549374,32.874116,47.763049,39.993111,53.985574,28.36864,43.420114,27.087165,42.706276



Per-country scores:


,country,turns,spBLEU,chrF++,BLEU,chrF
0,EG,1113,32.903189,46.892198,19.510451,49.894903
1,JO,1113,35.092749,49.225370,19.608431,52.790256
2,LB,1118,31.444152,45.774384,19.797027,48.505798
3,MA,1110,23.530143,39.376601,12.865149,42.236961
4,MR,1114,17.205693,33.791872,7.223466,37.942811
5,OM,1109,36.147408,49.843410,20.805106,53.247687
6,PS,1110,33.009470,47.549374,20.489823,50.817797
7,SA,1110,32.874116,47.763049,18.259737,51.802397
8,SY,1119,39.993111,53.985574,26.598272,56.940070
9,TN,1116,28.368640,43.420114,16.431024,46.081872



Saved artifacts:
Turn predictions: /home/mabdallah/alexandriax_mt_14d/inference_variants/05_retrieved_two_shot_with_participants/scored_turn_predictions.csv
Per-country metrics: /home/mabdallah/alexandriax_mt_14d/inference_variants/05_retrieved_two_shot_with_participants/per_country_official_metrics.csv
Official score row: /home/mabdallah/alexandriax_mt_14d/inference_variants/05_retrieved_two_shot_with_participants/official_leaderboard_score_row.csv
Metrics JSON: /home/mabdallah/alexandriax_mt_14d/inference_variants/05_retrieved_two_shot_with_participants/official_metrics.json
JSONL: /home/mabdallah/alexandriax_mt_14d/inference_variants/05_retrieved_two_shot_with_participants/predictions.jsonl
ZIP: /home/mabdallah/alexandriax_mt_14d/inference_variants/05_retrieved_two_shot_with_participants/submission_predictions.zip

✅ SUBMIT THIS ZIP:
/home/mabdallah/alexandriax_mt_14d/inference_variants/05_retrieved_two_shot_with_participants/submission_predictions.zip


### **Compare completed variants and build the best per-country mixed submission**

In [13]:
# ============================================================
# Cell 9 — Compare variants and build best-per-country mixture
#
# Selection:
#   Primary: country corpus spBLEU
#   Tie-break: country corpus chrF++
#
# One complete variant is selected per country.
# There is no reference-based per-turn oracle selection.
#
# Output:
#   inference_variants/90_mixed_best_variant_per_country/
#       turn_predictions.csv
#       country_variant_selection.csv
#       official metrics
#       predictions.jsonl
#       submission_predictions.zip
# ============================================================

CANDIDATE_VARIANT_NAMES = [
    "00_previous_official_control",
    "01_exact_training_parity",
    "02_metadata_no_shots",
    "03_retrieved_two_shot",
    "04_training_parity_with_participants",
     "05_retrieved_two_shot_with_participants",
]

completed_variant_records = []
completed_prediction_paths = {}

for variant_name in CANDIDATE_VARIANT_NAMES:
    variant_dir = (
        INFERENCE_VARIANTS_ROOT
        / variant_name
    )

    metric_path = (
        variant_dir
        / "per_country_official_metrics.csv"
    )

    prediction_path = (
        variant_dir
        / "turn_predictions.csv"
    )

    manifest_path = (
        variant_dir
        / "variant_manifest.json"
    )

    if not (
        metric_path.exists()
        and prediction_path.exists()
        and manifest_path.exists()
    ):
        print(
            "Skipping incomplete variant:",
            variant_name,
        )
        continue

    metric_df = pd.read_csv(metric_path)

    required_metric_columns = {
        "country",
        "spBLEU",
        "chrF++",
    }

    if not required_metric_columns.issubset(
        metric_df.columns
    ):
        print(
            "Skipping invalid metrics:",
            variant_name,
        )
        continue

    metric_df["variant_name"] = variant_name

    completed_variant_records.append(
        metric_df
    )

    completed_prediction_paths[
        variant_name
    ] = prediction_path

if not completed_variant_records:
    raise RuntimeError(
        "No completed variants were found."
    )

all_variant_country_metrics = pd.concat(
    completed_variant_records,
    ignore_index=True,
)

all_variant_country_metrics = (
    all_variant_country_metrics
    .sort_values(
        [
            "country",
            "spBLEU",
            "chrF++",
            "variant_name",
        ],
        ascending=[
            True,
            False,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

print("==============================")
print("All completed variant scores")
print("==============================")

display(
    all_variant_country_metrics[
        [
            "country",
            "variant_name",
            "turns",
            "spBLEU",
            "chrF++",
        ]
    ]
)

best_country_variant_df = (
    all_variant_country_metrics
    .groupby(
        "country",
        as_index=False,
        sort=True,
    )
    .first()
)

best_country_variant_df = (
    best_country_variant_df
    .rename(columns={
        "variant_name": "selected_variant",
        "spBLEU": "selection_dev_spBLEU",
        "chrF++": "selection_dev_chrF++",
    })
)

print("\nBest selected variant per country:")
display(
    best_country_variant_df[
        [
            "country",
            "selected_variant",
            "selection_dev_spBLEU",
            "selection_dev_chrF++",
        ]
    ]
)

selected_variant_map = dict(zip(
    best_country_variant_df["country"],
    best_country_variant_df[
        "selected_variant"
    ],
))

missing_selection_countries = (
    set(OFFICIAL_DEV_CONFIGS)
    - set(selected_variant_map)
)

if missing_selection_countries:
    raise RuntimeError(
        "No variant selected for countries: "
        f"{sorted(missing_selection_countries)}"
    )

# ------------------------------------------------------------
# Combine complete country predictions
# ------------------------------------------------------------

mixed_parts = []

for country in OFFICIAL_DEV_CONFIGS:
    selected_variant = (
        selected_variant_map[country]
    )

    prediction_path = (
        completed_prediction_paths[
            selected_variant
        ]
    )

    variant_prediction_df = pd.read_csv(
        prediction_path
    )

    variant_prediction_df["source_id"] = (
        variant_prediction_df[
            "source_id"
        ].astype(str)
    )

    country_ids = set(
        official_dev_df.loc[
            official_dev_df["config"] == country,
            "source_id",
        ].astype(str)
    )

    country_prediction_df = (
        variant_prediction_df[
            variant_prediction_df[
                "source_id"
            ].isin(country_ids)
        ]
        .copy()
    )

    if set(
        country_prediction_df[
            "source_id"
        ].astype(str)
    ) != country_ids:
        raise RuntimeError(
            f"Selected variant {selected_variant} "
            f"is incomplete for {country}."
        )

    country_prediction_df[
        "selected_from_variant"
    ] = selected_variant

    mixed_parts.append(
        country_prediction_df
    )

mixed_prediction_df = pd.concat(
    mixed_parts,
    ignore_index=True,
)

mixed_prediction_df = (
    mixed_prediction_df
    .sort_values(
        [
            "config",
            "conversation_id",
            "turn_order",
        ]
    )
    .reset_index(drop=True)
)

if len(mixed_prediction_df) != EXPECTED_DEV_TURNS:
    raise RuntimeError(
        "Mixed prediction count mismatch: "
        f"{len(mixed_prediction_df)}"
    )

if mixed_prediction_df[
    "source_id"
].duplicated().any():
    raise RuntimeError(
        "Duplicate source_id in mixed predictions."
    )

# ------------------------------------------------------------
# Independent mixed-variant folder
# ------------------------------------------------------------

MIXED_VARIANT_NAME = (
    "92_mixed_best_checkpoint_variant_per_country"
)
MIXED_VARIANT_DIR = (
    INFERENCE_VARIANTS_ROOT
    / MIXED_VARIANT_NAME
)

MIXED_VARIANT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

selection_payload = {
    "variant_name": MIXED_VARIANT_NAME,
    "selection_primary": (
        "per-country development corpus spBLEU"
    ),
    "selection_tiebreak": (
        "per-country development corpus chrF++"
    ),
    "checkpoint_step": CHECKPOINT_STEP,
    "country_to_variant": selected_variant_map,
    "candidate_variants": sorted(
        completed_prediction_paths
    ),
}

mixed_fingerprint = hashlib.sha256(
    json.dumps(
        selection_payload,
        ensure_ascii=False,
        sort_keys=True,
    ).encode("utf-8")
).hexdigest()

selection_payload[
    "variant_fingerprint"
] = mixed_fingerprint

with open(
    MIXED_VARIANT_DIR
    / "variant_manifest.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        selection_payload,
        f,
        ensure_ascii=False,
        indent=2,
    )

best_country_variant_df.to_csv(
    MIXED_VARIANT_DIR
    / "country_variant_selection.csv",
    index=False,
    encoding="utf-8-sig",
)

mixed_prediction_path = (
    MIXED_VARIANT_DIR
    / "turn_predictions.csv"
)

mixed_prediction_df.to_csv(
    mixed_prediction_path,
    index=False,
    encoding="utf-8-sig",
)

print("\nSaved mixed turn predictions:")
print(mixed_prediction_path)

# ------------------------------------------------------------
# Official scoring and submission packaging
# ------------------------------------------------------------

mixed_result = score_and_package_variant(
    prediction_df=mixed_prediction_df,
    variant_dir=MIXED_VARIANT_DIR,
    variant_name=MIXED_VARIANT_NAME,
    fingerprint=mixed_fingerprint,
)

All completed variant scores


,country,variant_name,turns,spBLEU,chrF++
0,EG,05_retrieved_two_shot_with_participants,1113,32.903189,46.892198
1,EG,01_exact_training_parity,1113,32.858879,46.987054
2,EG,03_retrieved_two_shot,1113,32.717798,46.715794
3,EG,04_training_parity_with_participants,1113,32.693573,46.924482
4,EG,02_metadata_no_shots,1113,32.644310,46.689370
...,...,...,...,...,...
61,YE,03_retrieved_two_shot,1118,26.998002,42.721976
62,YE,01_exact_training_parity,1118,26.822309,42.532265
63,YE,04_training_parity_with_participants,1118,26.756003,42.554430
64,YE,02_metadata_no_shots,1118,26.673491,42.433708



Best selected variant per country:


,country,selected_variant,selection_dev_spBLEU,selection_dev_chrF++
0,EG,05_retrieved_two_shot_with_participants,32.903189,46.892198
1,JO,03_retrieved_two_shot,35.301943,49.377895
2,LB,03_retrieved_two_shot,31.467859,45.831343
3,MA,05_retrieved_two_shot_with_participants,23.530143,39.376601
4,MR,03_retrieved_two_shot,17.367289,33.866626
5,OM,05_retrieved_two_shot_with_participants,36.147408,49.843410
6,PS,01_exact_training_parity,33.190460,47.623884
7,SA,03_retrieved_two_shot,33.012267,47.956335
8,SY,05_retrieved_two_shot_with_participants,39.993111,53.985574
9,TN,00_previous_official_control,28.509805,43.322662



Saved mixed turn predictions:
/home/mabdallah/alexandriax_mt_14d/inference_variants/90_mixed_best_variant_per_country/turn_predictions.csv

OFFICIAL-STYLE DEVELOPMENT RESULT
Variant: 90_mixed_best_variant_per_country
Checkpoint: 16600
Turns: 12250
Countries: 11
Average spBLEU (primary): 30.7737
Average chrF++: 45.5257

Leaderboard-like row:


,Variant,Checkpoint,Average spBLEU (primary),Average chrF++,EG spBLEU,EG chrF++,JO spBLEU,JO chrF++,LB spBLEU,LB chrF++,...,PS spBLEU,PS chrF++,SA spBLEU,SA chrF++,SY spBLEU,SY chrF++,TN spBLEU,TN chrF++,YE spBLEU,YE chrF++
0,90_mixed_best_variant_per_country,16600,30.773694,45.525709,32.903189,46.892198,35.301943,49.377895,31.467859,45.831343,...,33.19046,47.623884,33.012267,47.956335,39.993111,53.985574,28.509805,43.322662,27.087165,42.706276



Per-country scores:


,country,turns,spBLEU,chrF++,BLEU,chrF
0,EG,1113,32.903189,46.892198,19.510451,49.894903
1,JO,1113,35.301943,49.377895,19.819243,52.901916
2,LB,1118,31.467859,45.831343,19.797523,48.571026
3,MA,1110,23.530143,39.376601,12.865149,42.236961
4,MR,1114,17.367289,33.866626,7.179331,38.002673
5,OM,1109,36.147408,49.843410,20.805106,53.247687
6,PS,1110,33.190460,47.623884,20.580466,50.886702
7,SA,1110,33.012267,47.956335,18.305254,51.988206
8,SY,1119,39.993111,53.985574,26.598272,56.940070
9,TN,1116,28.509805,43.322662,16.643198,45.964899



Saved artifacts:
Turn predictions: /home/mabdallah/alexandriax_mt_14d/inference_variants/90_mixed_best_variant_per_country/scored_turn_predictions.csv
Per-country metrics: /home/mabdallah/alexandriax_mt_14d/inference_variants/90_mixed_best_variant_per_country/per_country_official_metrics.csv
Official score row: /home/mabdallah/alexandriax_mt_14d/inference_variants/90_mixed_best_variant_per_country/official_leaderboard_score_row.csv
Metrics JSON: /home/mabdallah/alexandriax_mt_14d/inference_variants/90_mixed_best_variant_per_country/official_metrics.json
JSONL: /home/mabdallah/alexandriax_mt_14d/inference_variants/90_mixed_best_variant_per_country/predictions.jsonl
ZIP: /home/mabdallah/alexandriax_mt_14d/inference_variants/90_mixed_best_variant_per_country/submission_predictions.zip

✅ SUBMIT THIS ZIP:
/home/mabdallah/alexandriax_mt_14d/inference_variants/90_mixed_best_variant_per_country/submission_predictions.zip


### **Checkpoint 16,500 with Variant 3**

In [15]:
# ============================================================
# Cell 10 — Checkpoint 16500 with strongest Variant 03
#
# Strategy:
#   - checkpoint 16500
#   - two semantically retrieved demonstrations
#   - current gender direction
#   - previous speaker labels
#   - no participants
#
# The cell temporarily changes CHECKPOINT_STEP/CHECKPOINT_PATH,
# runs the complete resumable experiment, then restores 16600.
#
# Output:
#   inference_variants/06_ckpt16500_retrieved_two_shot/
# ============================================================

_ORIGINAL_CHECKPOINT_STEP = CHECKPOINT_STEP
_ORIGINAL_CHECKPOINT_PATH = CHECKPOINT_PATH

try:
    CHECKPOINT_STEP = 16500
    CHECKPOINT_PATH = (
        TRAINING_RUN_DIR
        / f"checkpoint-{CHECKPOINT_STEP}"
    )

    if not CHECKPOINT_PATH.exists():
        raise FileNotFoundError(
            f"Checkpoint not found: {CHECKPOINT_PATH}"
        )

    if not (
        CHECKPOINT_PATH
        / "adapter_config.json"
    ).exists():
        raise FileNotFoundError(
            "adapter_config.json is missing from: "
            f"{CHECKPOINT_PATH}"
        )

    adapter_weight_candidates = [
        CHECKPOINT_PATH
        / "adapter_model.safetensors",

        CHECKPOINT_PATH
        / "adapter_model.bin",
    ]

    if not any(
        path.exists()
        for path in adapter_weight_candidates
    ):
        raise FileNotFoundError(
            "No adapter weights found inside: "
            f"{CHECKPOINT_PATH}"
        )

    print("==============================")
    print("Checkpoint 16500 inference")
    print("==============================")
    print("Checkpoint step:", CHECKPOINT_STEP)
    print("Checkpoint path:", CHECKPOINT_PATH)
    print("Strategy: retrieved two-shot")
    print("Participants: disabled")

    VARIANT_SPEC = {
        "variant_name": (
            "06_ckpt16500_retrieved_two_shot"
        ),
        "description": (
            "Checkpoint 16500 with two semantically retrieved "
            "same-country demonstrations, current speaker-to-"
            "addressee gender direction, previous speaker "
            "labels, and no participants metadata."
        ),
        "shot_mode": "retrieved",
        "use_direction": True,
        "use_previous_speakers": True,
        "use_participants": False,
    }

    variant_06_result = run_inference_variant(
        VARIANT_SPEC
    )

finally:
    # Restore checkpoint 16600 for all existing experiments.
    CHECKPOINT_STEP = _ORIGINAL_CHECKPOINT_STEP
    CHECKPOINT_PATH = _ORIGINAL_CHECKPOINT_PATH

    print("\nRestored default checkpoint:")
    print("CHECKPOINT_STEP:", CHECKPOINT_STEP)
    print("CHECKPOINT_PATH:", CHECKPOINT_PATH)

Checkpoint 16500 inference
Checkpoint step: 16500
Checkpoint path: /home/mabdallah/alexandriax_mt_14d/runs/nilechat3b_all14/nilechat3b_alexandria_all14_context3_complete2shot_all_group_r16_alpha32_3epochs_beam4_nonquant_server5090_v1/checkpoint-16500
Strategy: retrieved two-shot
Participants: disabled

START/RESUME INFERENCE VARIANT
Variant: 06_ckpt16500_retrieved_two_shot
Description: Checkpoint 16500 with two semantically retrieved same-country demonstrations, current speaker-to-addressee gender direction, previous speaker labels, and no participants metadata.
Folder: /home/mabdallah/alexandriax_mt_14d/inference_variants/06_ckpt16500_retrieved_two_shot
Fingerprint: 3bfa46341c25fbe518ffdbc58385e09c01a59049c612a665facbf8ea6ab91ca0
Shot mode: retrieved
Use direction: True
Use previous speakers: True
Use participants: False
Checkpoint: /home/mabdallah/alexandriax_mt_14d/runs/nilechat3b_all14/nilechat3b_alexandria_all14_context3_complete2shot_all_group_r16_alpha32_3epochs_beam4_nonquant_s

Loading weights:   0%|          | 0/435 [00:00<?, ?it/s]

06_ckpt16500_retrieved_two_shot:   0%|          | 0/12250 [00:00<?, ?it/s]

Saved: 12128 / 12250
Saved: 12178 / 12250
Saved: 12228 / 12250

Final prediction validation
Rows: 12250
Expected: 12250
Missing: 0
Extra: 0
Duplicates: 0
Empty: 0
Generation errors: 0

OFFICIAL-STYLE DEVELOPMENT RESULT
Variant: 06_ckpt16500_retrieved_two_shot
Checkpoint: 16500
Turns: 12250
Countries: 11
Average spBLEU (primary): 30.5509
Average chrF++: 45.5197

Leaderboard-like row:


,Variant,Checkpoint,Average spBLEU (primary),Average chrF++,EG spBLEU,EG chrF++,JO spBLEU,JO chrF++,LB spBLEU,LB chrF++,...,PS spBLEU,PS chrF++,SA spBLEU,SA chrF++,SY spBLEU,SY chrF++,TN spBLEU,TN chrF++,YE spBLEU,YE chrF++
0,06_ckpt16500_retrieved_two_shot,16500,30.550917,45.51967,32.21476,46.647475,34.632543,49.216474,31.343533,45.777729,...,33.418402,47.762506,33.166094,48.135685,39.915015,54.299895,28.407326,43.320124,27.380619,43.108824



Per-country scores:


,country,turns,spBLEU,chrF++,BLEU,chrF
0,EG,1113,32.214760,46.647475,18.842763,49.689530
1,JO,1113,34.632543,49.216474,19.151194,52.717139
2,LB,1118,31.343533,45.777729,19.752302,48.499898
3,MA,1110,22.709594,38.962070,12.193649,41.877302
4,MR,1114,16.820646,33.655636,7.045026,37.816991
5,OM,1109,36.051558,49.829957,20.644166,53.243055
6,PS,1110,33.418402,47.762506,20.699325,51.031341
7,SA,1110,33.166094,48.135685,18.903768,52.048449
8,SY,1119,39.915015,54.299895,26.324227,57.234543
9,TN,1116,28.407326,43.320124,16.669416,45.914258



Saved artifacts:
Turn predictions: /home/mabdallah/alexandriax_mt_14d/inference_variants/06_ckpt16500_retrieved_two_shot/scored_turn_predictions.csv
Per-country metrics: /home/mabdallah/alexandriax_mt_14d/inference_variants/06_ckpt16500_retrieved_two_shot/per_country_official_metrics.csv
Official score row: /home/mabdallah/alexandriax_mt_14d/inference_variants/06_ckpt16500_retrieved_two_shot/official_leaderboard_score_row.csv
Metrics JSON: /home/mabdallah/alexandriax_mt_14d/inference_variants/06_ckpt16500_retrieved_two_shot/official_metrics.json
JSONL: /home/mabdallah/alexandriax_mt_14d/inference_variants/06_ckpt16500_retrieved_two_shot/predictions.jsonl
ZIP: /home/mabdallah/alexandriax_mt_14d/inference_variants/06_ckpt16500_retrieved_two_shot/submission_predictions.zip

✅ SUBMIT THIS ZIP:
/home/mabdallah/alexandriax_mt_14d/inference_variants/06_ckpt16500_retrieved_two_shot/submission_predictions.zip

Restored default checkpoint:
CHECKPOINT_STEP: 16600
CHECKPOINT_PATH: /home/mabdallah

### **Mix of experts per country**

In [20]:
# ============================================================
# Checkpoint 16000 with strongest Variant 03
#
# Strategy:
#   - checkpoint 16000
#   - two semantically retrieved demonstrations
#   - current gender direction
#   - previous speaker labels
#   - no participants
#
# Output:
#   inference_variants/07_ckpt16000_retrieved_two_shot/
# ============================================================

_ORIGINAL_CHECKPOINT_STEP = CHECKPOINT_STEP
_ORIGINAL_CHECKPOINT_PATH = CHECKPOINT_PATH

try:
    CHECKPOINT_STEP = 16000
    CHECKPOINT_PATH = (
        TRAINING_RUN_DIR
        / f"checkpoint-{CHECKPOINT_STEP}"
    )

    if not CHECKPOINT_PATH.exists():
        raise FileNotFoundError(
            f"Checkpoint not found: {CHECKPOINT_PATH}"
        )

    if not (
        CHECKPOINT_PATH
        / "adapter_config.json"
    ).exists():
        raise FileNotFoundError(
            "adapter_config.json missing from: "
            f"{CHECKPOINT_PATH}"
        )

    adapter_weight_candidates = [
        CHECKPOINT_PATH / "adapter_model.safetensors",
        CHECKPOINT_PATH / "adapter_model.bin",
    ]

    if not any(
        path.exists()
        for path in adapter_weight_candidates
    ):
        raise FileNotFoundError(
            "No adapter weights found in: "
            f"{CHECKPOINT_PATH}"
        )

    print("==============================")
    print("Checkpoint 16000 inference")
    print("==============================")
    print("Checkpoint:", CHECKPOINT_PATH)
    print("Strategy: retrieved two-shot")
    print("Participants: disabled")

    VARIANT_SPEC = {
        "variant_name": (
            "07_ckpt16000_retrieved_two_shot"
        ),
        "description": (
            "Checkpoint 16000 with two semantically retrieved "
            "same-country demonstrations, current speaker-to-"
            "addressee gender direction, previous speaker "
            "labels, and no participants metadata."
        ),
        "shot_mode": "retrieved",
        "use_direction": True,
        "use_previous_speakers": True,
        "use_participants": False,
    }

    variant_07_result = run_inference_variant(
        VARIANT_SPEC
    )

finally:
    CHECKPOINT_STEP = _ORIGINAL_CHECKPOINT_STEP
    CHECKPOINT_PATH = _ORIGINAL_CHECKPOINT_PATH

    print("\nRestored default checkpoint:")
    print("CHECKPOINT_STEP:", CHECKPOINT_STEP)
    print("CHECKPOINT_PATH:", CHECKPOINT_PATH)

Checkpoint 16000 inference
Checkpoint: /home/mabdallah/alexandriax_mt_14d/runs/nilechat3b_all14/nilechat3b_alexandria_all14_context3_complete2shot_all_group_r16_alpha32_3epochs_beam4_nonquant_server5090_v1/checkpoint-16000
Strategy: retrieved two-shot
Participants: disabled

START/RESUME INFERENCE VARIANT
Variant: 07_ckpt16000_retrieved_two_shot
Description: Checkpoint 16000 with two semantically retrieved same-country demonstrations, current speaker-to-addressee gender direction, previous speaker labels, and no participants metadata.
Folder: /home/mabdallah/alexandriax_mt_14d/inference_variants/07_ckpt16000_retrieved_two_shot
Fingerprint: a06d542ed7fb7fe5c8d0b06ca5c62f3ceac610eb28c3fa719c358cff00cd9f96
Shot mode: retrieved
Use direction: True
Use previous speakers: True
Use participants: False
Checkpoint: /home/mabdallah/alexandriax_mt_14d/runs/nilechat3b_all14/nilechat3b_alexandria_all14_context3_complete2shot_all_group_r16_alpha32_3epochs_beam4_nonquant_server5090_v1/checkpoint-1600

,Variant,Checkpoint,Average spBLEU (primary),Average chrF++,EG spBLEU,EG chrF++,JO spBLEU,JO chrF++,LB spBLEU,LB chrF++,...,PS spBLEU,PS chrF++,SA spBLEU,SA chrF++,SY spBLEU,SY chrF++,TN spBLEU,TN chrF++,YE spBLEU,YE chrF++
0,07_ckpt16000_retrieved_two_shot,16000,30.652061,45.303998,32.764163,46.83995,35.159395,48.796993,31.701457,45.871215,...,32.921987,47.52135,32.848497,47.685507,39.821668,53.793826,29.285225,43.452865,26.620072,42.480666



Per-country scores:


,country,turns,spBLEU,chrF++,BLEU,chrF
0,EG,1113,32.764163,46.839950,19.246779,49.861679
1,JO,1113,35.159395,48.796993,19.356574,52.335074
2,LB,1118,31.701457,45.871215,20.021014,48.586333
3,MA,1110,23.003482,38.713336,12.415727,41.503732
4,MR,1114,17.380438,33.733013,7.100699,37.779153
5,OM,1109,35.666285,49.455254,20.254250,52.922760
6,PS,1110,32.921987,47.521350,20.445068,50.743675
7,SA,1110,32.848497,47.685507,18.396046,51.607516
8,SY,1119,39.821668,53.793826,26.385218,56.715341
9,TN,1116,29.285225,43.452865,17.062352,46.059624



Saved artifacts:
Turn predictions: /home/mabdallah/alexandriax_mt_14d/inference_variants/07_ckpt16000_retrieved_two_shot/scored_turn_predictions.csv
Per-country metrics: /home/mabdallah/alexandriax_mt_14d/inference_variants/07_ckpt16000_retrieved_two_shot/per_country_official_metrics.csv
Official score row: /home/mabdallah/alexandriax_mt_14d/inference_variants/07_ckpt16000_retrieved_two_shot/official_leaderboard_score_row.csv
Metrics JSON: /home/mabdallah/alexandriax_mt_14d/inference_variants/07_ckpt16000_retrieved_two_shot/official_metrics.json
JSONL: /home/mabdallah/alexandriax_mt_14d/inference_variants/07_ckpt16000_retrieved_two_shot/predictions.jsonl
ZIP: /home/mabdallah/alexandriax_mt_14d/inference_variants/07_ckpt16000_retrieved_two_shot/submission_predictions.zip

✅ SUBMIT THIS ZIP:
/home/mabdallah/alexandriax_mt_14d/inference_variants/07_ckpt16000_retrieved_two_shot/submission_predictions.zip

Restored default checkpoint:
CHECKPOINT_STEP: 16600
CHECKPOINT_PATH: /home/mabdallah

In [21]:
# ============================================================
# Best checkpoint × variant per country
#
# Candidates:
#   - all checkpoint-16600 variants
#   - checkpoint-16500 retrieved-two-shot variant
#
# Selection:
#   Primary: country-level corpus spBLEU
#   Tie-break: country-level corpus chrF++
#
# Output:
#   inference_variants/
#     92_mixed_best_checkpoint_variant_per_country/
#
#
# This preserves the previous 90_mixed... folder.
# ============================================================

CANDIDATE_VARIANT_NAMES = [
    "00_previous_official_control",
    "01_exact_training_parity",
    "02_metadata_no_shots",
    "03_retrieved_two_shot",
    "04_training_parity_with_participants",
    "05_retrieved_two_shot_with_participants",
    "06_ckpt16500_retrieved_two_shot",
    "07_ckpt16000_retrieved_two_shot",
]

candidate_metric_parts = []
candidate_prediction_paths = {}
candidate_manifest_data = {}

# ------------------------------------------------------------
# Load every completed candidate
# ------------------------------------------------------------

for variant_name in CANDIDATE_VARIANT_NAMES:
    variant_dir = (
        INFERENCE_VARIANTS_ROOT
        / variant_name
    )

    metric_path = (
        variant_dir
        / "per_country_official_metrics.csv"
    )

    prediction_path = (
        variant_dir
        / "turn_predictions.csv"
    )

    manifest_path = (
        variant_dir
        / "variant_manifest.json"
    )

    if not (
        metric_path.exists()
        and prediction_path.exists()
        and manifest_path.exists()
    ):
        print(
            "Skipping incomplete candidate:",
            variant_name,
        )
        continue

    with open(
        manifest_path,
        "r",
        encoding="utf-8",
    ) as f:
        manifest = json.load(f)

    candidate_checkpoint_step = (
        manifest.get(
            "checkpoint_step",
            "unknown",
        )
    )

    metric_df = pd.read_csv(metric_path)

    required_columns = {
        "country",
        "turns",
        "spBLEU",
        "chrF++",
    }

    missing_columns = (
        required_columns
        - set(metric_df.columns)
    )

    if missing_columns:
        print(
            "Skipping invalid candidate:",
            variant_name,
            "missing:",
            sorted(missing_columns),
        )
        continue

    metric_df["variant_name"] = variant_name
    metric_df["checkpoint_step"] = (
        candidate_checkpoint_step
    )

    candidate_metric_parts.append(
        metric_df
    )

    candidate_prediction_paths[
        variant_name
    ] = prediction_path

    candidate_manifest_data[
        variant_name
    ] = manifest

if not candidate_metric_parts:
    raise RuntimeError(
        "No completed checkpoint/variant candidates found."
    )

all_candidate_metrics = pd.concat(
    candidate_metric_parts,
    ignore_index=True,
)

# ------------------------------------------------------------
# Rank candidates independently for each country
# ------------------------------------------------------------

all_candidate_metrics = (
    all_candidate_metrics
    .sort_values(
        [
            "country",
            "spBLEU",
            "chrF++",
            "variant_name",
        ],
        ascending=[
            True,
            False,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

print("==============================")
print("All checkpoint × variant scores")
print("==============================")

display(
    all_candidate_metrics[
        [
            "country",
            "checkpoint_step",
            "variant_name",
            "turns",
            "spBLEU",
            "chrF++",
        ]
    ]
)

best_country_candidate_df = (
    all_candidate_metrics
    .groupby(
        "country",
        as_index=False,
        sort=True,
    )
    .first()
)

best_country_candidate_df = (
    best_country_candidate_df
    .rename(columns={
        "variant_name": "selected_variant",
        "checkpoint_step": "selected_checkpoint",
        "spBLEU": "selection_dev_spBLEU",
        "chrF++": "selection_dev_chrF++",
    })
)

print("\nBest checkpoint × variant per country:")

display(
    best_country_candidate_df[
        [
            "country",
            "selected_checkpoint",
            "selected_variant",
            "selection_dev_spBLEU",
            "selection_dev_chrF++",
        ]
    ]
)

selected_variant_map = dict(zip(
    best_country_candidate_df["country"],
    best_country_candidate_df[
        "selected_variant"
    ],
))

selected_checkpoint_map = dict(zip(
    best_country_candidate_df["country"],
    best_country_candidate_df[
        "selected_checkpoint"
    ],
))

missing_selection_countries = (
    set(OFFICIAL_DEV_CONFIGS)
    - set(selected_variant_map)
)

if missing_selection_countries:
    raise RuntimeError(
        "No selected candidate for countries: "
        f"{sorted(missing_selection_countries)}"
    )

# ------------------------------------------------------------
# Build mixed turn predictions
# ------------------------------------------------------------

mixed_prediction_parts = []

for country in OFFICIAL_DEV_CONFIGS:
    selected_variant = (
        selected_variant_map[country]
    )

    selected_checkpoint = (
        selected_checkpoint_map[country]
    )

    selected_prediction_path = (
        candidate_prediction_paths[
            selected_variant
        ]
    )

    selected_prediction_df = pd.read_csv(
        selected_prediction_path
    )

    selected_prediction_df["source_id"] = (
        selected_prediction_df[
            "source_id"
        ].astype(str)
    )

    expected_country_ids = set(
        official_dev_df.loc[
            official_dev_df["config"] == country,
            "source_id",
        ].astype(str)
    )

    selected_country_df = (
        selected_prediction_df[
            selected_prediction_df[
                "source_id"
            ].isin(expected_country_ids)
        ]
        .copy()
    )

    actual_country_ids = set(
        selected_country_df[
            "source_id"
        ].astype(str)
    )

    if actual_country_ids != expected_country_ids:
        missing_ids = (
            expected_country_ids
            - actual_country_ids
        )

        raise RuntimeError(
            f"Candidate {selected_variant} is "
            f"incomplete for {country}. "
            f"Missing IDs={len(missing_ids)}"
        )

    selected_country_df[
        "selected_from_variant"
    ] = selected_variant

    selected_country_df[
        "selected_from_checkpoint"
    ] = selected_checkpoint

    mixed_prediction_parts.append(
        selected_country_df
    )

mixed_prediction_df = pd.concat(
    mixed_prediction_parts,
    ignore_index=True,
)

mixed_prediction_df = (
    mixed_prediction_df
    .sort_values(
        [
            "config",
            "conversation_id",
            "turn_order",
        ]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Validate complete mixed result
# ------------------------------------------------------------

if len(mixed_prediction_df) != EXPECTED_DEV_TURNS:
    raise RuntimeError(
        "Mixed prediction count mismatch: "
        f"{len(mixed_prediction_df)}"
    )

if mixed_prediction_df[
    "source_id"
].duplicated().any():
    raise RuntimeError(
        "Duplicate source_id in mixed predictions."
    )

mixed_prediction_df["prediction"] = (
    mixed_prediction_df["prediction"]
    .fillna("")
    .astype(str)
    .str.strip()
)

if int(
    (
        mixed_prediction_df[
            "prediction"
        ] == ""
    ).sum()
):
    raise RuntimeError(
        "Empty predictions in mixed result."
    )

# ------------------------------------------------------------
# Create new independent mixed folder
# ------------------------------------------------------------

MIXED_VARIANT_NAME = (
    "92_mixed_best_checkpoint_variant_per_country"
)

MIXED_VARIANT_DIR = (
    INFERENCE_VARIANTS_ROOT
    / MIXED_VARIANT_NAME
)

MIXED_VARIANT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

selection_records = (
    best_country_candidate_df[
        [
            "country",
            "selected_checkpoint",
            "selected_variant",
            "selection_dev_spBLEU",
            "selection_dev_chrF++",
        ]
    ]
    .to_dict("records")
)

selection_payload = {
    "variant_name": MIXED_VARIANT_NAME,
    "selection_type": (
        "best checkpoint and inference variant "
        "per country"
    ),
    "selection_primary": (
        "country corpus spBLEU"
    ),
    "selection_tiebreak": (
        "country corpus chrF++"
    ),
    "candidate_variants": (
        CANDIDATE_VARIANT_NAMES
    ),
    "country_selections": selection_records,
    "created_at": time.strftime(
        "%Y-%m-%d %H:%M:%S"
    ),
}

mixed_fingerprint = hashlib.sha256(
    json.dumps(
        selection_payload,
        ensure_ascii=False,
        sort_keys=True,
    ).encode("utf-8")
).hexdigest()

selection_payload[
    "variant_fingerprint"
] = mixed_fingerprint

with open(
    MIXED_VARIANT_DIR
    / "variant_manifest.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        selection_payload,
        f,
        ensure_ascii=False,
        indent=2,
    )

best_country_candidate_df.to_csv(
    MIXED_VARIANT_DIR
    / "country_checkpoint_variant_selection.csv",
    index=False,
    encoding="utf-8-sig",
)

mixed_prediction_path = (
    MIXED_VARIANT_DIR
    / "turn_predictions.csv"
)

mixed_prediction_df.to_csv(
    mixed_prediction_path,
    index=False,
    encoding="utf-8-sig",
)

print("\nSaved checkpoint × variant mixture:")
print(mixed_prediction_path)

# ------------------------------------------------------------
# Score/package with an accurate mixed-checkpoint label
# ------------------------------------------------------------

_ORIGINAL_CHECKPOINT_STEP = CHECKPOINT_STEP
_ORIGINAL_CHECKPOINT_PATH = CHECKPOINT_PATH

try:
    CHECKPOINT_STEP = "mixed_16000_16500_16600"
    CHECKPOINT_PATH = TRAINING_RUN_DIR

    mixed_checkpoint_result = (
        score_and_package_variant(
            prediction_df=mixed_prediction_df,
            variant_dir=MIXED_VARIANT_DIR,
            variant_name=MIXED_VARIANT_NAME,
            fingerprint=mixed_fingerprint,
        )
    )

finally:
    CHECKPOINT_STEP = _ORIGINAL_CHECKPOINT_STEP
    CHECKPOINT_PATH = _ORIGINAL_CHECKPOINT_PATH

    print("\nRestored default checkpoint globals:")
    print("CHECKPOINT_STEP:", CHECKPOINT_STEP)
    print("CHECKPOINT_PATH:", CHECKPOINT_PATH)

All checkpoint × variant scores


,country,checkpoint_step,variant_name,turns,spBLEU,chrF++
0,EG,16600,05_retrieved_two_shot_with_participants,1113,32.903189,46.892198
1,EG,16600,01_exact_training_parity,1113,32.858879,46.987054
2,EG,16000,07_ckpt16000_retrieved_two_shot,1113,32.764163,46.839950
3,EG,16600,03_retrieved_two_shot,1113,32.717798,46.715794
4,EG,16600,04_training_parity_with_participants,1113,32.693573,46.924482
...,...,...,...,...,...,...
83,YE,16600,01_exact_training_parity,1118,26.822309,42.532265
84,YE,16600,04_training_parity_with_participants,1118,26.756003,42.554430
85,YE,16600,02_metadata_no_shots,1118,26.673491,42.433708
86,YE,16000,07_ckpt16000_retrieved_two_shot,1118,26.620072,42.480666



Best checkpoint × variant per country:


,country,selected_checkpoint,selected_variant,selection_dev_spBLEU,selection_dev_chrF++
0,EG,16600,05_retrieved_two_shot_with_participants,32.903189,46.892198
1,JO,16600,03_retrieved_two_shot,35.301943,49.377895
2,LB,16000,07_ckpt16000_retrieved_two_shot,31.701457,45.871215
3,MA,16600,05_retrieved_two_shot_with_participants,23.530143,39.376601
4,MR,16000,07_ckpt16000_retrieved_two_shot,17.380438,33.733013
5,OM,16600,05_retrieved_two_shot_with_participants,36.147408,49.843410
6,PS,16500,06_ckpt16500_retrieved_two_shot,33.418402,47.762506
7,SA,16500,06_ckpt16500_retrieved_two_shot,33.166094,48.135685
8,SY,16600,05_retrieved_two_shot_with_participants,39.993111,53.985574
9,TN,16000,07_ckpt16000_retrieved_two_shot,29.285225,43.452865



Saved checkpoint × variant mixture:
/home/mabdallah/alexandriax_mt_14d/inference_variants/92_mixed_best_checkpoint_variant_per_country/turn_predictions.csv

OFFICIAL-STYLE DEVELOPMENT RESULT
Variant: 92_mixed_best_checkpoint_variant_per_country
Checkpoint: mixed_16000_16500_16600
Turns: 12250
Countries: 11
Average spBLEU (primary): 30.9280
Average chrF++: 45.5945

Leaderboard-like row:


,Variant,Checkpoint,Average spBLEU (primary),Average chrF++,EG spBLEU,EG chrF++,JO spBLEU,JO chrF++,LB spBLEU,LB chrF++,...,PS spBLEU,PS chrF++,SA spBLEU,SA chrF++,SY spBLEU,SY chrF++,TN spBLEU,TN chrF++,YE spBLEU,YE chrF++
0,92_mixed_best_checkpoint_variant_per_country,mixed_16000_16500_16600,30.928003,45.594526,32.903189,46.892198,35.301943,49.377895,31.701457,45.871215,...,33.418402,47.762506,33.166094,48.135685,39.993111,53.985574,29.285225,43.452865,27.380619,43.108824



Per-country scores:


,country,turns,spBLEU,chrF++,BLEU,chrF
0,EG,1113,32.903189,46.892198,19.510451,49.894903
1,JO,1113,35.301943,49.377895,19.819243,52.901916
2,LB,1118,31.701457,45.871215,20.021014,48.586333
3,MA,1110,23.530143,39.376601,12.865149,42.236961
4,MR,1114,17.380438,33.733013,7.100699,37.779153
5,OM,1109,36.147408,49.843410,20.805106,53.247687
6,PS,1110,33.418402,47.762506,20.699325,51.031341
7,SA,1110,33.166094,48.135685,18.903768,52.048449
8,SY,1119,39.993111,53.985574,26.598272,56.940070
9,TN,1116,29.285225,43.452865,17.062352,46.059624



Saved artifacts:
Turn predictions: /home/mabdallah/alexandriax_mt_14d/inference_variants/92_mixed_best_checkpoint_variant_per_country/scored_turn_predictions.csv
Per-country metrics: /home/mabdallah/alexandriax_mt_14d/inference_variants/92_mixed_best_checkpoint_variant_per_country/per_country_official_metrics.csv
Official score row: /home/mabdallah/alexandriax_mt_14d/inference_variants/92_mixed_best_checkpoint_variant_per_country/official_leaderboard_score_row.csv
Metrics JSON: /home/mabdallah/alexandriax_mt_14d/inference_variants/92_mixed_best_checkpoint_variant_per_country/official_metrics.json
JSONL: /home/mabdallah/alexandriax_mt_14d/inference_variants/92_mixed_best_checkpoint_variant_per_country/predictions.jsonl
ZIP: /home/mabdallah/alexandriax_mt_14d/inference_variants/92_mixed_best_checkpoint_variant_per_country/submission_predictions.zip

✅ SUBMIT THIS ZIP:
/home/mabdallah/alexandriax_mt_14d/inference_variants/92_mixed_best_checkpoint_variant_per_country/submission_prediction

### **Oracle Analysis Predictions**

In [16]:
# ============================================================
# ORACLE-ZIP — Package the reference-leaking DEV oracle
#
# This is a DEV diagnostic, not a hidden-TEST system.
# ============================================================

ORACLE_SOURCE_DIR = (
    INFERENCE_VARIANTS_ROOT
    / "_oracle_analysis_v1"
)

ORACLE_CSV = (
    ORACLE_SOURCE_DIR
    / (
        "oracle_coordinate_corpus_spbleu_"
        "diagnostic_predictions.csv"
    )
)

if not ORACLE_CSV.exists():
    raise FileNotFoundError(
        f"Oracle file not found: {ORACLE_CSV}\n"
        "Run the coordinate-oracle analysis first."
    )

oracle_source_df = pd.read_csv(
    ORACLE_CSV
)

required_columns = {
    "source_id",
    "prediction",
}

missing_columns = (
    required_columns
    - set(oracle_source_df.columns)
)

if missing_columns:
    raise RuntimeError(
        "Oracle file is missing: "
        f"{sorted(missing_columns)}"
    )

oracle_source_df["source_id"] = (
    oracle_source_df["source_id"]
    .astype(str)
)

oracle_source_df["prediction"] = (
    oracle_source_df["prediction"]
    .fillna("")
    .astype(str)
    .str.strip()
)

expected_ids = set(
    official_dev_df[
        "source_id"
    ].astype(str)
)

actual_ids = set(
    oracle_source_df[
        "source_id"
    ]
)

if actual_ids != expected_ids:
    raise RuntimeError(
        "Oracle IDs do not match DEV: "
        f"missing={len(expected_ids - actual_ids)}, "
        f"extra={len(actual_ids - expected_ids)}"
    )

if oracle_source_df[
    "source_id"
].duplicated().any():
    raise RuntimeError(
        "Duplicate source_id in oracle file."
    )

if oracle_source_df[
    "prediction"
].eq("").any():
    raise RuntimeError(
        "Empty oracle prediction."
    )

oracle_package_df = (
    oracle_source_df[
        [
            "source_id",
            "prediction",
        ]
    ].copy()
)

oracle_package_df[
    "raw_prediction"
] = oracle_package_df["prediction"]

oracle_package_df[
    "clean_prediction"
] = oracle_package_df["prediction"]

oracle_package_df[
    "generation_error"
] = ""

ORACLE_PACKAGE_NAME = (
    "99_REFERENCE_LEAKING_"
    "DEV_COORDINATE_ORACLE"
)

ORACLE_PACKAGE_DIR = (
    INFERENCE_VARIANTS_ROOT
    / ORACLE_PACKAGE_NAME
)

ORACLE_PACKAGE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

oracle_manifest = {
    "variant_name": ORACLE_PACKAGE_NAME,
    "diagnostic_only": True,
    "reference_leaking": True,
    "source_file": str(ORACLE_CSV),
    "selection": (
        "per-turn exact reference-based "
        "corpus-spBLEU coordinate oracle"
    ),
    "warning": (
        "Not reproducible on hidden TEST "
        "without references."
    ),
    "turns": int(
        len(oracle_package_df)
    ),
    "created_at": time.strftime(
        "%Y-%m-%d %H:%M:%S"
    ),
}

oracle_fingerprint = (
    hashlib.sha256(
        json.dumps(
            oracle_manifest,
            ensure_ascii=False,
            sort_keys=True,
        ).encode("utf-8")
    ).hexdigest()
)

oracle_manifest[
    "variant_fingerprint"
] = oracle_fingerprint

with open(
    ORACLE_PACKAGE_DIR
    / "variant_manifest.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        oracle_manifest,
        handle,
        ensure_ascii=False,
        indent=2,
    )

oracle_package_df.to_csv(
    ORACLE_PACKAGE_DIR
    / "turn_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)

_original_step = CHECKPOINT_STEP
_original_path = CHECKPOINT_PATH

try:
    CHECKPOINT_STEP = (
        "REFERENCE_LEAKING_DEV_ORACLE"
    )

    CHECKPOINT_PATH = (
        ORACLE_SOURCE_DIR
    )

    oracle_package_result = (
        score_and_package_variant(
            prediction_df=(
                oracle_package_df
            ),
            variant_dir=(
                ORACLE_PACKAGE_DIR
            ),
            variant_name=(
                ORACLE_PACKAGE_NAME
            ),
            fingerprint=(
                oracle_fingerprint
            ),
        )
    )

finally:
    CHECKPOINT_STEP = _original_step
    CHECKPOINT_PATH = _original_path

print(
    "\nREFERENCE-LEAKING "
    "DEV ORACLE ZIP:"
)

print(
    ORACLE_PACKAGE_DIR
    / "submission_predictions.zip"
)


OFFICIAL-STYLE DEVELOPMENT RESULT
Variant: 99_REFERENCE_LEAKING_DEV_COORDINATE_ORACLE
Checkpoint: REFERENCE_LEAKING_DEV_ORACLE
Turns: 12250
Countries: 11
Average spBLEU (primary): 36.0127
Average chrF++: 49.3783

Leaderboard-like row:


,Variant,Checkpoint,Average spBLEU (primary),Average chrF++,EG spBLEU,EG chrF++,JO spBLEU,JO chrF++,LB spBLEU,LB chrF++,...,PS spBLEU,PS chrF++,SA spBLEU,SA chrF++,SY spBLEU,SY chrF++,TN spBLEU,TN chrF++,YE spBLEU,YE chrF++
0,99_REFERENCE_LEAKING_DEV_COORDINATE_ORACLE,REFERENCE_LEAKING_DEV_ORACLE,36.012678,49.378289,38.322313,50.69639,41.247103,53.497325,36.916171,49.775979,...,38.020698,51.317832,37.952262,51.616942,45.92152,58.279403,33.772705,47.096758,32.283325,46.727053



Per-country scores:


,country,turns,spBLEU,chrF++,BLEU,chrF
0,EG,1113,38.322313,50.696390,23.830333,53.512917
1,JO,1113,41.247103,53.497325,24.761291,56.683809
2,LB,1118,36.916171,49.775979,24.239096,52.336172
3,MA,1110,28.489118,43.190678,16.552205,45.902426
4,MR,1114,22.066587,37.309126,9.722501,41.343766
5,OM,1109,41.147652,53.653689,25.037382,56.833701
6,PS,1110,38.020698,51.317832,24.711946,54.393426
7,SA,1110,37.952262,51.616942,22.661116,55.421084
8,SY,1119,45.921520,58.279403,32.090706,60.939931
9,TN,1116,33.772705,47.096758,20.678740,49.622941



Saved artifacts:
Turn predictions: /home/mabdallah/alexandriax_mt_14d/inference_variants/99_REFERENCE_LEAKING_DEV_COORDINATE_ORACLE/scored_turn_predictions.csv
Per-country metrics: /home/mabdallah/alexandriax_mt_14d/inference_variants/99_REFERENCE_LEAKING_DEV_COORDINATE_ORACLE/per_country_official_metrics.csv
Official score row: /home/mabdallah/alexandriax_mt_14d/inference_variants/99_REFERENCE_LEAKING_DEV_COORDINATE_ORACLE/official_leaderboard_score_row.csv
Metrics JSON: /home/mabdallah/alexandriax_mt_14d/inference_variants/99_REFERENCE_LEAKING_DEV_COORDINATE_ORACLE/official_metrics.json
JSONL: /home/mabdallah/alexandriax_mt_14d/inference_variants/99_REFERENCE_LEAKING_DEV_COORDINATE_ORACLE/predictions.jsonl
ZIP: /home/mabdallah/alexandriax_mt_14d/inference_variants/99_REFERENCE_LEAKING_DEV_COORDINATE_ORACLE/submission_predictions.zip

✅ SUBMIT THIS ZIP:
/home/mabdallah/alexandriax_mt_14d/inference_variants/99_REFERENCE_LEAKING_DEV_COORDINATE_ORACLE/submission_predictions.zip

REFEREN

## **New Experiment**

In [17]:
# ============================================================
# I0 — Exact LoRA interpolation configuration
#
# No GPU/model loading in this cell.
# ============================================================

from copy import deepcopy

from safetensors.torch import (
    load_file as safe_load_file,
    save_file as safe_save_file,
)

INTERP_CHECKPOINTS = {
    16000: (
        TRAINING_RUN_DIR
        / "checkpoint-16000"
    ),
    16500: (
        TRAINING_RUN_DIR
        / "checkpoint-16500"
    ),
    16600: (
        TRAINING_RUN_DIR
        / "checkpoint-16600"
    ),
}

# One fixed experiment.
INTERP_WEIGHTS = {
    16000: 0.15,
    16500: 0.35,
    16600: 0.50,
}

if abs(
    sum(INTERP_WEIGHTS.values())
    - 1.0
) > 1e-12:
    raise RuntimeError(
        "Interpolation weights must sum to 1."
    )

if (
    set(INTERP_WEIGHTS)
    != set(INTERP_CHECKPOINTS)
):
    raise RuntimeError(
        "Checkpoint and weight keys differ."
    )


def interp_weight_path(
    checkpoint_dir,
):
    safetensor_path = (
        checkpoint_dir
        / "adapter_model.safetensors"
    )

    binary_path = (
        checkpoint_dir
        / "adapter_model.bin"
    )

    if safetensor_path.exists():
        return safetensor_path

    if binary_path.exists():
        return binary_path

    raise FileNotFoundError(
        f"No adapter weights inside "
        f"{checkpoint_dir}"
    )


def interp_load_weights(
    checkpoint_dir,
):
    path = interp_weight_path(
        checkpoint_dir
    )

    if path.suffix == ".safetensors":
        return safe_load_file(
            str(path),
            device="cpu",
        )

    try:
        return torch.load(
            path,
            map_location="cpu",
            weights_only=True,
        )

    except TypeError:
        return torch.load(
            path,
            map_location="cpu",
        )


interp_configs = {}
interp_weight_paths = {}

for (
    step,
    checkpoint_dir,
) in INTERP_CHECKPOINTS.items():

    config_path = (
        checkpoint_dir
        / "adapter_config.json"
    )

    if not checkpoint_dir.exists():
        raise FileNotFoundError(
            f"Missing checkpoint: "
            f"{checkpoint_dir}"
        )

    if not config_path.exists():
        raise FileNotFoundError(
            f"Missing adapter config: "
            f"{config_path}"
        )

    with open(
        config_path,
        "r",
        encoding="utf-8",
    ) as handle:
        interp_configs[
            step
        ] = json.load(handle)

    interp_weight_paths[
        step
    ] = interp_weight_path(
        checkpoint_dir
    )

reference_step = 16600

reference_config = (
    interp_configs[
        reference_step
    ]
)

compatibility_fields = [
    "peft_type",
    "task_type",
    "target_modules",
    "fan_in_fan_out",
    "bias",
    "use_dora",
]

for (
    step,
    config,
) in interp_configs.items():

    for field in compatibility_fields:
        if (
            config.get(field)
            != reference_config.get(field)
        ):
            raise RuntimeError(
                "Incompatible adapter field "
                f"{field}: "
                f"step {step}="
                f"{config.get(field)!r}, "
                f"step {reference_step}="
                f"{reference_config.get(field)!r}"
            )

    if config.get(
        "use_dora",
        False,
    ):
        raise RuntimeError(
            "DoRA interpolation is not "
            "supported by this cell."
        )

INTERP_ADAPTER_NAME = (
    "interp_exact_delta_r48_"
    "s16000w015_"
    "s16500w035_"
    "s16600w050"
)

INTERP_ADAPTER_DIR = (
    TRAINING_RUN_DIR
    / "interpolated_adapters"
    / INTERP_ADAPTER_NAME
)

print("Checkpoint adapters:")

for step in sorted(
    INTERP_CHECKPOINTS
):
    print(
        f"step {step}: "
        f"weight="
        f"{INTERP_WEIGHTS[step]:.2f}, "
        f"r="
        f"{interp_configs[step].get('r')}, "
        f"alpha="
        f"{interp_configs[step].get('lora_alpha')}, "
        f"file="
        f"{interp_weight_paths[step]}"
    )

print(
    "Output adapter:",
    INTERP_ADAPTER_DIR,
)

Checkpoint adapters:
step 16000: weight=0.15, r=16, alpha=32, file=/home/mabdallah/alexandriax_mt_14d/runs/nilechat3b_all14/nilechat3b_alexandria_all14_context3_complete2shot_all_group_r16_alpha32_3epochs_beam4_nonquant_server5090_v1/checkpoint-16000/adapter_model.safetensors
step 16500: weight=0.35, r=16, alpha=32, file=/home/mabdallah/alexandriax_mt_14d/runs/nilechat3b_all14/nilechat3b_alexandria_all14_context3_complete2shot_all_group_r16_alpha32_3epochs_beam4_nonquant_server5090_v1/checkpoint-16500/adapter_model.safetensors
step 16600: weight=0.50, r=16, alpha=32, file=/home/mabdallah/alexandriax_mt_14d/runs/nilechat3b_all14/nilechat3b_alexandria_all14_context3_complete2shot_all_group_r16_alpha32_3epochs_beam4_nonquant_server5090_v1/checkpoint-16600/adapter_model.safetensors
Output adapter: /home/mabdallah/alexandriax_mt_14d/runs/nilechat3b_all14/nilechat3b_alexandria_all14_context3_complete2shot_all_group_r16_alpha32_3epochs_beam4_nonquant_server5090_v1/interpolated_adapters/interp

In [18]:
# ============================================================
# I1 — Build exact weighted sum of complete LoRA deltas
#
# A_mix = vertical concatenation of A_i
# B_mix = horizontal concatenation of:
#         weight_i × original_scaling_i × B_i
#
# New scaling is one, so:
#
# B_mix @ A_mix
# = sum(weight_i × scaling_i × B_i @ A_i)
#
# This is exact and does not average A/B separately.
# ============================================================

def interp_pair_b_key(a_key):
    if ".lora_A." not in a_key:
        raise ValueError(
            f"Not a LoRA-A key: {a_key}"
        )

    return a_key.replace(
        ".lora_A.",
        ".lora_B.",
        1,
    )


def interp_module_name(a_key):
    return a_key.split(
        ".lora_A.",
        1,
    )[0]


def interp_pattern_value(
    pattern,
    module_name,
    default,
):
    if not pattern:
        return default

    matches = [
        (key, value)
        for key, value in pattern.items()
        if module_name.endswith(
            str(key)
        )
    ]

    if not matches:
        return default

    matches.sort(
        key=lambda item: len(
            str(item[0])
        ),
        reverse=True,
    )

    return matches[0][1]


def interp_original_scaling(
    config,
    module_name,
    actual_rank,
):
    configured_rank = int(
        interp_pattern_value(
            config.get(
                "rank_pattern",
                {},
            ),
            module_name,
            config.get(
                "r",
                actual_rank,
            ),
        )
    )

    if (
        configured_rank
        != int(actual_rank)
    ):
        raise RuntimeError(
            "Rank/config mismatch for "
            f"{module_name}: "
            f"tensor={actual_rank}, "
            f"config={configured_rank}"
        )

    alpha = float(
        interp_pattern_value(
            config.get(
                "alpha_pattern",
                {},
            ),
            module_name,
            config.get(
                "lora_alpha",
                configured_rank,
            ),
        )
    )

    if config.get(
        "use_rslora",
        False,
    ):
        return (
            alpha
            / math.sqrt(
                configured_rank
            )
        )

    return (
        alpha
        / configured_rank
    )


step_weights = {
    step: interp_load_weights(
        checkpoint_dir
    )
    for (
        step,
        checkpoint_dir,
    ) in INTERP_CHECKPOINTS.items()
}

reference_keys = set(
    step_weights[
        reference_step
    ]
)

for (
    step,
    state,
) in step_weights.items():

    if set(state) != reference_keys:
        raise RuntimeError(
            "Adapter tensor keys differ "
            f"at step {step}: "
            f"missing="
            f"{len(reference_keys - set(state))}, "
            f"extra="
            f"{len(set(state) - reference_keys)}"
        )

a_keys = sorted(
    key
    for key in reference_keys
    if ".lora_A." in key
)

b_keys = sorted(
    key
    for key in reference_keys
    if ".lora_B." in key
)

expected_b_keys = {
    interp_pair_b_key(key)
    for key in a_keys
}

if not a_keys:
    raise RuntimeError(
        "No LoRA A tensors found."
    )

if set(b_keys) != expected_b_keys:
    raise RuntimeError(
        "LoRA A/B pairing failed."
    )

supported_keys = (
    set(a_keys)
    | set(b_keys)
)

unexpected_keys = sorted(
    reference_keys
    - supported_keys
)

if unexpected_keys:
    raise RuntimeError(
        "Unsupported non-LoRA tensors:\n - "
        + "\n - ".join(
            unexpected_keys[:30]
        )
    )

interpolated_state = {}
new_ranks = set()

ordered_steps = list(
    INTERP_WEIGHTS
)

for a_key in tqdm(
    a_keys,
    desc="Interpolating LoRA modules",
):
    b_key = interp_pair_b_key(
        a_key
    )

    module_name = (
        interp_module_name(
            a_key
        )
    )

    a_blocks = []
    b_blocks = []

    reference_input_shape = None
    reference_output_shape = None

    for step in ordered_steps:
        a_tensor = (
            step_weights[
                step
            ][a_key]
            .float()
        )

        b_tensor = (
            step_weights[
                step
            ][b_key]
            .float()
        )

        if (
            a_tensor.ndim != 2
            or b_tensor.ndim != 2
        ):
            raise RuntimeError(
                "Expected 2-D LoRA "
                f"tensors for {module_name}"
            )

        if (
            b_tensor.shape[1]
            != a_tensor.shape[0]
        ):
            raise RuntimeError(
                "A/B rank mismatch for "
                f"{module_name}, "
                f"step {step}"
            )

        input_shape = (
            a_tensor.shape[1],
        )

        output_shape = (
            b_tensor.shape[0],
        )

        if reference_input_shape is None:
            reference_input_shape = (
                input_shape
            )

            reference_output_shape = (
                output_shape
            )

        elif (
            input_shape
            != reference_input_shape
            or output_shape
            != reference_output_shape
        ):
            raise RuntimeError(
                "Input/output dimensions "
                f"differ for {module_name}, "
                f"step {step}"
            )

        original_scale = (
            interp_original_scaling(
                interp_configs[
                    step
                ],
                module_name,
                actual_rank=(
                    a_tensor.shape[0]
                ),
            )
        )

        a_blocks.append(
            a_tensor
        )

        b_blocks.append(
            float(
                INTERP_WEIGHTS[
                    step
                ]
            )
            * original_scale
            * b_tensor
        )

    mixed_a = torch.cat(
        a_blocks,
        dim=0,
    ).contiguous()

    mixed_b = torch.cat(
        b_blocks,
        dim=1,
    ).contiguous()

    if (
        mixed_b.shape[1]
        != mixed_a.shape[0]
    ):
        raise RuntimeError(
            "Mixed A/B rank mismatch "
            f"for {module_name}"
        )

    new_ranks.add(
        int(
            mixed_a.shape[0]
        )
    )

    interpolated_state[
        a_key
    ] = mixed_a.to(
        step_weights[
            reference_step
        ][a_key].dtype
    )

    interpolated_state[
        b_key
    ] = mixed_b.to(
        step_weights[
            reference_step
        ][b_key].dtype
    )

if len(new_ranks) != 1:
    raise RuntimeError(
        "Mixed modules have "
        "different ranks: "
        f"{sorted(new_ranks)}"
    )

INTERP_NEW_RANK = int(
    next(iter(new_ranks))
)

# Validate exact algebra on multiple modules.
validation_keys = a_keys[
    :min(
        8,
        len(a_keys),
    )
]

maximum_validation_error = 0.0

for (
    test_number,
    a_key,
) in enumerate(validation_keys):

    b_key = interp_pair_b_key(
        a_key
    )

    input_width = int(
        interpolated_state[
            a_key
        ].shape[1]
    )

    generator = (
        torch.Generator(
            device="cpu"
        )
    )

    generator.manual_seed(
        3407 + test_number
    )

    probe = torch.randn(
        input_width,
        3,
        generator=generator,
    )

    expected = None

    for step in ordered_steps:
        a_tensor = (
            step_weights[
                step
            ][a_key]
            .float()
        )

        b_tensor = (
            step_weights[
                step
            ][b_key]
            .float()
        )

        scale = (
            interp_original_scaling(
                interp_configs[
                    step
                ],
                interp_module_name(
                    a_key
                ),
                actual_rank=(
                    a_tensor.shape[0]
                ),
            )
        )

        contribution = (
            float(
                INTERP_WEIGHTS[
                    step
                ]
            )
            * scale
            * (
                b_tensor
                @ (
                    a_tensor
                    @ probe
                )
            )
        )

        expected = (
            contribution
            if expected is None
            else (
                expected
                + contribution
            )
        )

    actual = (
        interpolated_state[
            b_key
        ].float()
        @ (
            interpolated_state[
                a_key
            ].float()
            @ probe
        )
    )

    error = float(
        (
            actual
            - expected
        )
        .abs()
        .max()
    )

    maximum_validation_error = max(
        maximum_validation_error,
        error,
    )

if (
    maximum_validation_error
    > 5e-4
):
    raise RuntimeError(
        "Interpolation algebra "
        "validation failed: "
        f"{maximum_validation_error}"
    )

new_config = deepcopy(
    reference_config
)

new_config["r"] = (
    INTERP_NEW_RANK
)

# New scaling = alpha/r = 1.
new_config["lora_alpha"] = (
    INTERP_NEW_RANK
)

new_config["rank_pattern"] = {}
new_config["alpha_pattern"] = {}
new_config["use_rslora"] = False
new_config["inference_mode"] = True

INTERP_ADAPTER_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

safe_save_file(
    {
        key: value.contiguous()
        for key, value in (
            interpolated_state.items()
        )
    },
    str(
        INTERP_ADAPTER_DIR
        / "adapter_model.safetensors"
    ),
    metadata={
        "format": "pt",
    },
)

with open(
    INTERP_ADAPTER_DIR
    / "adapter_config.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        new_config,
        handle,
        ensure_ascii=False,
        indent=2,
    )

interpolation_manifest = {
    "name": INTERP_ADAPTER_NAME,
    "method": (
        "exact weighted sum of "
        "complete scaled LoRA deltas"
    ),
    "weights": {
        str(step): weight
        for (
            step,
            weight,
        ) in INTERP_WEIGHTS.items()
    },
    "source_checkpoints": {
        str(step): str(path)
        for (
            step,
            path,
        ) in INTERP_CHECKPOINTS.items()
    },
    "source_ranks": {
        str(step): int(
            interp_configs[
                step
            ]["r"]
        )
        for step in (
            INTERP_CHECKPOINTS
        )
    },
    "output_rank": (
        INTERP_NEW_RANK
    ),
    "output_alpha": (
        INTERP_NEW_RANK
    ),
    "algebra_validation_max_abs_error": (
        maximum_validation_error
    ),
    "created_at": time.strftime(
        "%Y-%m-%d %H:%M:%S"
    ),
}

with open(
    INTERP_ADAPTER_DIR
    / "interpolation_manifest.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        interpolation_manifest,
        handle,
        ensure_ascii=False,
        indent=2,
    )

print(
    "Exact interpolation adapter created."
)

print(
    "Rank:",
    INTERP_NEW_RANK,
)

print(
    "Alpha:",
    INTERP_NEW_RANK,
)

print(
    "Validation max error:",
    maximum_validation_error,
)

print(
    "Adapter:",
    INTERP_ADAPTER_DIR,
)

del step_weights
del interpolated_state

gc.collect()

Interpolating LoRA modules:   0%|          | 0/252 [00:00<?, ?it/s]

Exact interpolation adapter created.
Rank: 48
Alpha: 48
Validation max error: 2.384185791015625e-07
Adapter: /home/mabdallah/alexandriax_mt_14d/runs/nilechat3b_all14/nilechat3b_alexandria_all14_context3_complete2shot_all_group_r16_alpha32_3epochs_beam4_nonquant_server5090_v1/interpolated_adapters/interp_exact_delta_r48_s16000w015_s16500w035_s16600w050


1092

In [20]:
# ============================================================
# I2 — Run/resume interpolation inference
# Includes temporary legacy-tokenizer compatibility repair
# ============================================================

import json
import inspect
from transformers import AutoTokenizer

INTERP_VARIANT_NAME = (
    "08_interp_015_035_050_retrieved_two_shot"
)

INTERP_VARIANT_SPEC = {
    "variant_name": INTERP_VARIANT_NAME,
    "description": (
        "Exact LoRA-delta interpolation of checkpoints "
        "16000/16500/16600 with weights 0.15/0.35/0.50, "
        "using retrieved two-shot inference, gender direction, "
        "previous speaker labels and no participants."
    ),
    "shot_mode": "retrieved",
    "use_direction": True,
    "use_previous_speakers": True,
    "use_participants": False,
}

# ------------------------------------------------------------
# Validate interpolated adapter
# ------------------------------------------------------------

if not (
    INTERP_ADAPTER_DIR / "adapter_config.json"
).exists():
    raise FileNotFoundError(
        "Interpolated adapter_config.json is missing. Run I1."
    )

if not (
    INTERP_ADAPTER_DIR / "adapter_model.safetensors"
).exists():
    raise FileNotFoundError(
        "Interpolated adapter weights are missing. Run I1."
    )

# ------------------------------------------------------------
# Inspect the legacy tokenizer configuration
# ------------------------------------------------------------

tokenizer_config_path = (
    BASE_MODEL_DIR / "tokenizer_config.json"
)

legacy_extra_special_tokens = None

if tokenizer_config_path.exists():
    with open(
        tokenizer_config_path,
        "r",
        encoding="utf-8",
    ) as file:
        tokenizer_config = json.load(file)

    legacy_extra_special_tokens = tokenizer_config.get(
        "extra_special_tokens"
    )

    print(
        "Stored extra_special_tokens type:",
        type(legacy_extra_special_tokens).__name__,
    )

# New Transformers expects extra_special_tokens to be a dict.
needs_tokenizer_repair = isinstance(
    legacy_extra_special_tokens,
    list,
)

if needs_tokenizer_repair:
    print(
        "Applying temporary tokenizer compatibility repair."
    )
else:
    print(
        "Tokenizer configuration does not require list repair."
    )

# ------------------------------------------------------------
# Temporarily wrap AutoTokenizer.from_pretrained
#
# This changes nothing on disk. It only prevents the new
# Transformers version from treating the legacy list as a dict.
# ------------------------------------------------------------

_original_auto_tokenizer_descriptor = inspect.getattr_static(
    AutoTokenizer,
    "from_pretrained",
)

_original_auto_tokenizer_loader = (
    AutoTokenizer.from_pretrained
)


def _compatible_auto_tokenizer_loader(
    cls,
    pretrained_model_name_or_path,
    *args,
    **kwargs,
):
    if needs_tokenizer_repair:
        kwargs["extra_special_tokens"] = {}

    tokenizer = _original_auto_tokenizer_loader(
        pretrained_model_name_or_path,
        *args,
        **kwargs,
    )

    # Decoder-only generation must use left padding.
    tokenizer.padding_side = "left"

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    return tokenizer


# ------------------------------------------------------------
# Switch the notebook loader to the interpolated adapter
# ------------------------------------------------------------

_original_checkpoint_step = CHECKPOINT_STEP
_original_checkpoint_path = CHECKPOINT_PATH

try:
    AutoTokenizer.from_pretrained = classmethod(
        _compatible_auto_tokenizer_loader
    )

    CHECKPOINT_STEP = (
        "interp_16000x015_16500x035_16600x050_r48"
    )
    CHECKPOINT_PATH = INTERP_ADAPTER_DIR

    print("Inference checkpoint:", CHECKPOINT_PATH)

    interpolation_result = run_inference_variant(
        INTERP_VARIANT_SPEC
    )

finally:
    # Restore the actual AutoTokenizer method.
    AutoTokenizer.from_pretrained = (
        _original_auto_tokenizer_descriptor
    )

    # Restore the notebook's default checkpoint.
    CHECKPOINT_STEP = _original_checkpoint_step
    CHECKPOINT_PATH = _original_checkpoint_path

    print(
        "Restored default checkpoint:",
        CHECKPOINT_STEP,
        CHECKPOINT_PATH,
    )

# ------------------------------------------------------------
# Confirm output
# ------------------------------------------------------------

INTERP_VARIANT_DIR = (
    INFERENCE_VARIANTS_ROOT
    / INTERP_VARIANT_NAME
)

INTERP_SUBMISSION_ZIP = (
    INTERP_VARIANT_DIR
    / "submission_predictions.zip"
)

print("\nPURE INTERPOLATION SUBMISSION ZIP:")
print(INTERP_SUBMISSION_ZIP)

if not INTERP_SUBMISSION_ZIP.exists():
    raise FileNotFoundError(
        "Inference did not complete; rerun this same cell "
        "to resume from saved predictions."
    )

Stored extra_special_tokens type: list
Applying temporary tokenizer compatibility repair.
Inference checkpoint: /home/mabdallah/alexandriax_mt_14d/runs/nilechat3b_all14/nilechat3b_alexandria_all14_context3_complete2shot_all_group_r16_alpha32_3epochs_beam4_nonquant_server5090_v1/interpolated_adapters/interp_exact_delta_r48_s16000w015_s16500w035_s16600w050

START/RESUME INFERENCE VARIANT
Variant: 08_interp_015_035_050_retrieved_two_shot
Description: Exact LoRA-delta interpolation of checkpoints 16000/16500/16600 with weights 0.15/0.35/0.50, using retrieved two-shot inference, gender direction, previous speaker labels and no participants.
Folder: /home/mabdallah/alexandriax_mt_14d/inference_variants/08_interp_015_035_050_retrieved_two_shot
Fingerprint: 45a8d6d127149be94ed8219a3de3ade1e4a4c075d5b28ba7bf90e2bab52feccf
Shot mode: retrieved
Use direction: True
Use previous speakers: True
Use participants: False
Checkpoint: /home/mabdallah/alexandriax_mt_14d/runs/nilechat3b_all14/nilechat3b_al

`torch_dtype` is deprecated! Use `dtype` instead!


08_interp_015_035_050_retrieved_two_shot:   0%|          | 0/12250 [00:00<?, ?it/s]

Saved: 50 / 12250
Saved: 100 / 12250
Saved: 150 / 12250
Saved: 200 / 12250
Saved: 250 / 12250
Saved: 300 / 12250
Saved: 350 / 12250
Saved: 400 / 12250
Saved: 450 / 12250
Saved: 500 / 12250
Saved: 550 / 12250
Saved: 600 / 12250
Saved: 650 / 12250
Saved: 700 / 12250
Saved: 750 / 12250
Saved: 800 / 12250
Saved: 850 / 12250
Saved: 900 / 12250
Saved: 950 / 12250
Saved: 1000 / 12250
Saved: 1050 / 12250
Saved: 1100 / 12250
Saved: 1150 / 12250
Saved: 1200 / 12250
Saved: 1400 / 12250
Saved: 1450 / 12250
Saved: 1500 / 12250
Saved: 1550 / 12250
Saved: 1600 / 12250
Saved: 1650 / 12250
Saved: 1700 / 12250
Saved: 1750 / 12250
Saved: 1800 / 12250
Saved: 1850 / 12250
Saved: 1900 / 12250
Saved: 1950 / 12250
Saved: 2000 / 12250
Saved: 2050 / 12250
Saved: 2100 / 12250
Saved: 2150 / 12250
Saved: 2200 / 12250
Saved: 2250 / 12250
Saved: 2300 / 12250
Saved: 2350 / 12250
Saved: 2400 / 12250
Saved: 2450 / 12250
Saved: 2500 / 12250
Saved: 2550 / 12250
Saved: 2600 / 12250
Saved: 2650 / 12250
Saved: 2700 / 12250


,Variant,Checkpoint,Average spBLEU (primary),Average chrF++,EG spBLEU,EG chrF++,JO spBLEU,JO chrF++,LB spBLEU,LB chrF++,...,PS spBLEU,PS chrF++,SA spBLEU,SA chrF++,SY spBLEU,SY chrF++,TN spBLEU,TN chrF++,YE spBLEU,YE chrF++
0,08_interp_015_035_050_retrieved_two_shot,interp_16000x015_16500x035_16600x050_r48,28.786161,44.618648,30.29901,45.998529,32.572102,48.465367,29.553025,44.893548,...,31.23003,46.670248,31.293672,47.350206,37.317895,52.740236,26.947404,42.483192,25.239596,42.085014



Per-country scores:


,country,turns,spBLEU,chrF++,BLEU,chrF
0,EG,1113,30.299010,45.998529,17.612511,49.059877
1,JO,1113,32.572102,48.465367,17.974420,51.978641
2,LB,1118,29.553025,44.893548,18.475480,47.691145
3,MA,1110,22.057886,38.560869,12.142524,41.380838
4,MR,1114,16.352146,33.226560,7.144303,37.267391
5,OM,1109,33.785007,48.331355,18.621393,51.859281
6,PS,1110,31.230030,46.670248,19.038691,49.923097
7,SA,1110,31.293672,47.350206,17.602792,51.312765
8,SY,1119,37.317895,52.740236,24.218440,55.769025
9,TN,1116,26.947404,42.483192,15.214053,45.164382



Saved artifacts:
Turn predictions: /home/mabdallah/alexandriax_mt_14d/inference_variants/08_interp_015_035_050_retrieved_two_shot/scored_turn_predictions.csv
Per-country metrics: /home/mabdallah/alexandriax_mt_14d/inference_variants/08_interp_015_035_050_retrieved_two_shot/per_country_official_metrics.csv
Official score row: /home/mabdallah/alexandriax_mt_14d/inference_variants/08_interp_015_035_050_retrieved_two_shot/official_leaderboard_score_row.csv
Metrics JSON: /home/mabdallah/alexandriax_mt_14d/inference_variants/08_interp_015_035_050_retrieved_two_shot/official_metrics.json
JSONL: /home/mabdallah/alexandriax_mt_14d/inference_variants/08_interp_015_035_050_retrieved_two_shot/predictions.jsonl
ZIP: /home/mabdallah/alexandriax_mt_14d/inference_variants/08_interp_015_035_050_retrieved_two_shot/submission_predictions.zip

✅ SUBMIT THIS ZIP:
/home/mabdallah/alexandriax_mt_14d/inference_variants/08_interp_015_035_050_retrieved_two_shot/submission_predictions.zip
Restored default check

In [21]:
# ============================================================
# I3 — Compare interpolation with system 92 per country
#
# Uses country-level DEV selection, not per-turn oracle routing.
# ============================================================

INTERP_VARIANT_DIR = (
    INFERENCE_VARIANTS_ROOT
    / INTERP_VARIANT_NAME
)

SYSTEM92_NAME = (
    "92_mixed_best_checkpoint_variant_per_country"
)

SYSTEM92_DIR = (
    INFERENCE_VARIANTS_ROOT
    / SYSTEM92_NAME
)

interp_metric_path = (
    INTERP_VARIANT_DIR
    / "per_country_official_metrics.csv"
)

interp_prediction_path = (
    INTERP_VARIANT_DIR
    / "turn_predictions.csv"
)

base_metric_path = (
    SYSTEM92_DIR
    / "per_country_official_metrics.csv"
)

base_prediction_path = (
    SYSTEM92_DIR
    / "turn_predictions.csv"
)

for required_path in [
    interp_metric_path,
    interp_prediction_path,
    base_metric_path,
    base_prediction_path,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            "Missing completed artifact: "
            f"{required_path}"
        )

interp_metrics = pd.read_csv(
    interp_metric_path
)

base_metrics = pd.read_csv(
    base_metric_path
)

comparison_df = (
    base_metrics[
        [
            "country",
            "turns",
            "spBLEU",
            "chrF++",
        ]
    ]
    .rename(
        columns={
            "spBLEU": (
                "system92_spBLEU"
            ),
            "chrF++": (
                "system92_chrFpp"
            ),
        }
    )
    .merge(
        interp_metrics[
            [
                "country",
                "spBLEU",
                "chrF++",
            ]
        ].rename(
            columns={
                "spBLEU": (
                    "interpolation_spBLEU"
                ),
                "chrF++": (
                    "interpolation_chrFpp"
                ),
            }
        ),
        on="country",
        how="inner",
        validate="one_to_one",
    )
)

comparison_df[
    "spBLEU_gain"
] = (
    comparison_df[
        "interpolation_spBLEU"
    ]
    - comparison_df[
        "system92_spBLEU"
    ]
)

comparison_df[
    "chrFpp_gain"
] = (
    comparison_df[
        "interpolation_chrFpp"
    ]
    - comparison_df[
        "system92_chrFpp"
    ]
)

INTERP_MIN_COUNTRY_GAIN = 0.02

comparison_df[
    "use_interpolation"
] = (
    comparison_df[
        "spBLEU_gain"
    ] > INTERP_MIN_COUNTRY_GAIN
) | (
    comparison_df[
        "spBLEU_gain"
    ].gt(0.0)
    & comparison_df[
        "spBLEU_gain"
    ].abs().le(
        INTERP_MIN_COUNTRY_GAIN
    )
    & comparison_df[
        "chrFpp_gain"
    ].gt(0.0)
)

print(
    "Interpolation versus system 92:"
)

display(
    comparison_df.sort_values(
        "spBLEU_gain",
        ascending=False,
    )
)

selected_countries = set(
    comparison_df.loc[
        comparison_df[
            "use_interpolation"
        ],
        "country",
    ].astype(str)
)

if not selected_countries:
    print(
        "\nNo country improved. "
        "Keep system 92."
    )

    print(
        "The pure interpolation ZIP "
        "remains a diagnostic."
    )

else:
    base_predictions = pd.read_csv(
        base_prediction_path
    )

    interp_predictions = pd.read_csv(
        interp_prediction_path
    )

    for frame in [
        base_predictions,
        interp_predictions,
    ]:
        frame["source_id"] = (
            frame["source_id"]
            .astype(str)
        )

    prediction_columns = [
        "prediction",
        "raw_prediction",
        "clean_prediction",
        "generation_error",
    ]

    for column in prediction_columns:
        if (
            column
            not in base_predictions.columns
        ):
            base_predictions[column] = (
                base_predictions[
                    "prediction"
                ]
                if column
                != "generation_error"
                else ""
            )

        if (
            column
            not in interp_predictions.columns
        ):
            interp_predictions[column] = (
                interp_predictions[
                    "prediction"
                ]
                if column
                != "generation_error"
                else ""
            )

    interp_lookup = (
        interp_predictions
        .set_index("source_id")
    )

    final_predictions = (
        base_predictions.copy()
    )

    country_column = (
        "config"
        if "config"
        in final_predictions.columns
        else "country"
    )

    replace_mask = (
        final_predictions[
            country_column
        ]
        .astype(str)
        .isin(selected_countries)
    )

    replace_ids = (
        final_predictions.loc[
            replace_mask,
            "source_id",
        ]
    )

    for column in prediction_columns:
        final_predictions.loc[
            replace_mask,
            column,
        ] = (
            replace_ids
            .map(
                interp_lookup[
                    column
                ]
            )
            .to_numpy()
        )

    if (
        "selected_from_variant"
        not in final_predictions.columns
    ):
        final_predictions[
            "selected_from_variant"
        ] = SYSTEM92_NAME

    final_predictions.loc[
        replace_mask,
        "selected_from_variant",
    ] = INTERP_VARIANT_NAME

    MIX_NAME = (
        "95_system92_plus_"
        "interpolation_per_country"
    )

    MIX_DIR = (
        INFERENCE_VARIANTS_ROOT
        / MIX_NAME
    )

    MIX_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    comparison_df.to_csv(
        MIX_DIR
        / "country_interpolation_selection.csv",
        index=False,
        encoding="utf-8-sig",
    )

    final_predictions.to_csv(
        MIX_DIR
        / "turn_predictions.csv",
        index=False,
        encoding="utf-8-sig",
    )

    mix_manifest = {
        "variant_name": MIX_NAME,
        "selection_type": (
            "per-country comparison of "
            "system92 and exact interpolation"
        ),
        "base_system": SYSTEM92_NAME,
        "interpolation_variant": (
            INTERP_VARIANT_NAME
        ),
        "interpolation_adapter": str(
            INTERP_ADAPTER_DIR
        ),
        "interpolation_weights": {
            str(step): weight
            for (
                step,
                weight,
            ) in INTERP_WEIGHTS.items()
        },
        "selected_interpolation_countries": (
            sorted(
                selected_countries
            )
        ),
        "minimum_country_spBLEU_gain": (
            INTERP_MIN_COUNTRY_GAIN
        ),
        "created_at": time.strftime(
            "%Y-%m-%d %H:%M:%S"
        ),
    }

    mix_fingerprint = (
        hashlib.sha256(
            json.dumps(
                mix_manifest,
                ensure_ascii=False,
                sort_keys=True,
            ).encode("utf-8")
        ).hexdigest()
    )

    mix_manifest[
        "variant_fingerprint"
    ] = mix_fingerprint

    with open(
        MIX_DIR
        / "variant_manifest.json",
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            mix_manifest,
            handle,
            ensure_ascii=False,
            indent=2,
        )

    _original_step = CHECKPOINT_STEP
    _original_path = CHECKPOINT_PATH

    try:
        CHECKPOINT_STEP = (
            "system92_plus_"
            "exact_interpolation"
        )

        CHECKPOINT_PATH = (
            INTERP_ADAPTER_DIR
        )

        interpolation_mix_result = (
            score_and_package_variant(
                prediction_df=(
                    final_predictions
                ),
                variant_dir=MIX_DIR,
                variant_name=MIX_NAME,
                fingerprint=(
                    mix_fingerprint
                ),
            )
        )

    finally:
        CHECKPOINT_STEP = (
            _original_step
        )

        CHECKPOINT_PATH = (
            _original_path
        )

    print(
        "\nINTERPOLATION-ENHANCED ZIP:"
    )

    print(
        MIX_DIR
        / "submission_predictions.zip"
    )

Interpolation versus system 92:


,country,turns,system92_spBLEU,system92_chrFpp,interpolation_spBLEU,interpolation_chrFpp,spBLEU_gain,chrFpp_gain,use_interpolation
4,MR,1114,17.380438,33.733013,16.352146,33.226560,-1.028293,-0.506453,False
3,MA,1110,23.530143,39.376601,22.057886,38.560869,-1.472257,-0.815732,False
7,SA,1110,33.166094,48.135685,31.293672,47.350206,-1.872423,-0.785479,False
10,YE,1118,27.380619,43.108824,25.239596,42.085014,-2.141024,-1.023811,False
2,LB,1118,31.701457,45.871215,29.553025,44.893548,-2.148432,-0.977666,False
6,PS,1110,33.418402,47.762506,31.230030,46.670248,-2.188371,-1.092257,False
9,TN,1116,29.285225,43.452865,26.947404,42.483192,-2.337821,-0.969673,False
5,OM,1109,36.147408,49.843410,33.785007,48.331355,-2.362400,-1.512054,False
0,EG,1113,32.903189,46.892198,30.299010,45.998529,-2.604179,-0.893669,False
8,SY,1119,39.993111,53.985574,37.317895,52.740236,-2.675216,-1.245338,False



No country improved. Keep system 92.
The pure interpolation ZIP remains a diagnostic.
